# (a) Tạo dataset YOLO-seg bằng pseudo-label tự động (ngập đường phố)

Chạy trên Kaggle với Internet bật. Notebook này chỉ tạo `/kaggle/working/dataset`; tải folder đó thành Kaggle Dataset rồi add vào notebook train riêng.

Input cần add: `pthttm-teachers`, `water-dataset`, `flood-yolo-seg-dataset`. Tuỳ chọn: Roadway Flooding và **ít nhất một nguồn ảnh đường phố không ngập** (thư mục `non_flooded/`, `dry_road/`, `wet_road/`, `reflection/`, `puddle/`… hoặc khai báo `NEGATIVE_INPUT_DIRS`). Notebook chỉ đọc ảnh từ dataset cũ và **bỏ qua tuyệt đối mọi `labels/`, `.txt`, `data.yaml`** trong đó.

- **Flood**: mask thật của WaterDataset (trừ thư mục sông/hồ/biển trong `WATER_EXCLUDE_DIRS`)/Roadway, hoặc WaterNet khi đồng thuận với SegFormer (IoU ≥ 0.5). Ảnh không có bối cảnh đường phố hoặc nước là sông/hồ/biển bị loại.
- **Road**: lớp `road` của SegFormer ADE20K (tra theo `id2label`), trừ flood và xe, có kiểm tra confidence, diện tích, độ vụn và vị trí; road không đáng tin thì ảnh bị loại, không xuất nhãn mù quáng.
- **Car**: level từ `best_car_YOLOv11.pt` (conf ≥ 0.5) + mask `yolo11x-seg` (IoU ≥ 0.5); không ghép được thì bỏ.
- **Ảnh âm**: chỉ từ nguồn được tuyển chọn là không ngập, nhãn rỗng khi không có class hợp lệ; không bao giờ xoá flood mà teacher thấy để giả làm ảnh âm.
- **Split**: group theo video/nguồn (thư mục, prefix tên file, ảnh gần trùng), bảo đảm class có mặt ở train và val; test = WaterDataset có mask thật, **chỉ có nhãn flood**.
- Cuối notebook có bảng thống kê và cổng chất lượng: không đạt thì ghi `NOT_READY_FOR_TRAINING.txt`.

Class order cố định: `0 flood`, `1 road`, `2 car_L0`, `3 car_L1`, `4 car_L2`, `5 car_L3`, `6 car_L4`.


In [ ]:
# Cell 1: khám phá Kaggle Input, tìm weights và cặp ảnh/mask mà không giả định mount path.
from pathlib import Path
import re, os, hashlib, random, json, csv, shutil, math, collections, time

# Trial mode: maximum images sampled from each source. Set None to process all images.
LIMIT = 50
print(f'LIMIT = {LIMIT}', '(chạy thử)' if LIMIT is not None else '(dataset đầy đủ)')
INPUT_ROOT = Path('/kaggle/input')
if not INPUT_ROOT.exists():
    raise FileNotFoundError('Không thấy /kaggle/input. Hãy chạy notebook trong Kaggle và Add ba input dataset.')

def show_tree(root, maxdepth=4):
    print(f'\nCây thư mục {root} (maxdepth={maxdepth}):')
    for p in sorted(root.rglob('*')):
        try: depth=len(p.relative_to(root).parts)
        except ValueError: continue
        if depth > maxdepth: continue
        print('  '*depth + ('[D] ' if p.is_dir() else '[F] ') + p.name)
show_tree(INPUT_ROOT, 4)

def norm(s): return re.sub(r'[^a-z0-9]', '', str(s).lower())
def dirs_named(root, names):
    wanted={norm(x) for x in names}
    return [p for p in root.rglob('*') if p.is_dir() and norm(p.name) in wanted]
def files_named(root, filename):
    return [p for p in root.rglob(filename) if p.is_file()]

water_weights=files_named(INPUT_ROOT,'link_efficientb4_model.pth')
car_weights=files_named(INPUT_ROOT,'best_car_YOLOv11.pt')
if not water_weights: raise FileNotFoundError('Không tìm thấy **/link_efficientb4_model.pth. Kiểm tra pthttm-teachers đã được Add vào notebook.')
if not car_weights: raise FileNotFoundError('Không tìm thấy **/best_car_YOLOv11.pt. Kiểm tra pthttm-teachers đã được Add vào notebook.')
WATER_WEIGHTS=water_weights[0]; CAR_LEVEL_WEIGHTS=car_weights[0]
print('[FOUND] WaterNet weight candidates:',*[str(p) for p in water_weights],sep='\n  ')
print('[FOUND] Car-level weight candidates:',*[str(p) for p in car_weights],sep='\n  ')

# Locate WaterDataset (either dataset mount root or nested archive extraction).
water_roots=dirs_named(INPUT_ROOT,['WaterDataset'])
if not water_roots: water_roots=dirs_named(INPUT_ROOT,['water-dataset'])
if not water_roots: raise FileNotFoundError('Không tìm thấy thư mục WaterDataset/water-dataset trong /kaggle/input.')
WATER_ROOT=water_roots[-1] if any(norm(p.name)=='waterdataset' for p in water_roots) else water_roots[0]
old_roots=[p for p in INPUT_ROOT.rglob('*') if p.is_dir() and norm(p.name)=='floodyolosegdataset']
if not old_roots: raise FileNotFoundError('Không tìm thấy thư mục flood_yolo_seg_dataset (hoặc tên tương đương) trong /kaggle/input.')
OLD_ROOT=old_roots[0]
roadway_roots=[p for p in INPUT_ROOT.rglob('*') if p.is_dir() and norm(p.name)=='roadwayfloodingimagedataset']
ROADWAY_ROOT=roadway_roots[0] if roadway_roots else None
print('\nFound WaterNet:',WATER_WEIGHTS)
print('Found car-level model:',CAR_LEVEL_WEIGHTS)
print('WaterDataset root:',WATER_ROOT)
print('Old image dataset root (labels will be ignored):',OLD_ROOT)
print('Roadway Flooding dataset root:',ROADWAY_ROOT if ROADWAY_ROOT else 'NOT ATTACHED (skip; add Kaggle input to include)')

# Discover likely image/mask locations immediately and fail early on a wrong input.
IMAGE_EXTS={'.jpg','.jpeg','.png','.bmp','.webp'}
MASK_EXTS=IMAGE_EXTS|{'.tif','.tiff','.npy'}
MASK_HINTS=('mask','groundtruth','annotation','label','gt')
water_image_dirs=[p for p in WATER_ROOT.rglob('*') if p.is_dir() and 'image' in norm(p.name) and not any(t in norm(p.name) for t in MASK_HINTS) and any(q.is_file() and q.suffix.lower() in IMAGE_EXTS for q in p.rglob('*'))]
water_mask_dirs=[p for p in WATER_ROOT.rglob('*') if p.is_dir() and any(t in norm(p.name) for t in MASK_HINTS)]
water_mask_files=[p for p in WATER_ROOT.rglob('*') if p.is_file() and p.suffix.lower() in MASK_EXTS and (any(t in norm(x) for x in p.parts for t in MASK_HINTS) or any(p.stem.lower().endswith(s) for s in ('_mask','-mask','_label','-label','_gt','-gt','_water','-water')))]
if not water_image_dirs: raise FileNotFoundError(f'WaterDataset image directory not found under {WATER_ROOT}; expected train_images/JPEGImages or an equivalent images folder.')
if not water_mask_dirs and not water_mask_files: raise FileNotFoundError(f'WaterDataset mask directory/files not found under {WATER_ROOT}; inspect printed tree and add the correct dataset version.')
print('WaterDataset image dirs:',*[str(p) for p in water_image_dirs[:20]],sep='\n  ')
print('WaterDataset mask dirs:',*[str(p) for p in water_mask_dirs[:20]],sep='\n  ')
print('WaterDataset mask-like files:',len(water_mask_files))
print('Candidate WaterDataset image roots:',len(water_image_dirs),'| mask roots:',len(water_mask_dirs))

## 0. Cấu hình

Mọi ngưỡng (road, bối cảnh đường phố, ảnh âm, split, xe, cổng chất lượng) nằm ở cell dưới, có giải thích ngắn. Giá trị mặc định chưa được hiệu chỉnh trên dữ liệu thật; chạy `LIMIT = 50`, xem `preview/`, `preview_road/`, `preview_road_rejected/`, `auto_label_manifest.csv`, `dataset/QUALITY_REPORT.json` rồi chỉnh.


In [ ]:
# Cell cấu hình: mọi ngưỡng của pipeline. Giá trị mặc định là điểm xuất phát hợp lý, CHƯA hiệu chỉnh
# trên dữ liệu thật. Chạy LIMIT=50, xem preview*/, auto_label_manifest.csv, QUALITY_REPORT.json rồi chỉnh.

# ======================= ROAD (SegFormer ADE20K) =======================
# Teacher road/water: SegFormer ADE20K. B0 (mIoU ADE ~37) bỏ sót mặt đường ướt/phản chiếu (vd. 33b392ea0af731f2:
# chỉ 4.1% ảnh là argmax=road). B5 (mIoU ~51) cùng họ, cùng nhãn ADE20K, chậm hơn nhưng vẫn nhanh trên GPU Kaggle.
# Đổi lại 'nvidia/segformer-b0-finetuned-ade-512-512' nếu thiếu GPU/RAM.
SEGFORMER_MODEL='nvidia/segformer-b5-finetuned-ade-640-640'
# Road = phần mặt đường nhìn thấy được. Chỉ lớp 'road' của ADE20K (tra theo model.config.id2label, không
# hard-code id); không gộp sidewalk/path/floor/earth/roof. Flood, xe và vùng đệm quanh chúng bị trừ khỏi road.
ROAD_CLASS_NAMES=('road',)
# (1) Pixel: argmax = road VÀ P(road) >= ROAD_PIXEL_CONF VÀ P(water+sea+river+lake) <= ROAD_MAX_WATER_PROB.
ROAD_PIXEL_CONF=0.50
ROAD_MAX_WATER_PROB=0.20
# (2) Làm sạch: kernel = ROAD_MORPH_FRAC * cạnh ngắn (>= 3 px); lấp lỗ kín nhỏ hơn REGION_FILL_HOLE_FRAC ảnh.
ROAD_MORPH_FRAC=0.01
REGION_FILL_HOLE_FRAC=0.002
# (3) road = road_pred - dilate(flood, ROAD_FLOOD_GAP_PX) - dilate(xe, ROAD_VEHICLE_GAP_PX).
#     Đệm flood > 2*REGION_POLY_EPS_PX + 1 để polygon đã đơn giản hoá của road và flood không chạm nhau.
ROAD_FLOOD_GAP_PX=4
ROAD_VEHICLE_GAP_PX=2
REGION_POLY_EPS_PX=1.0
# (4) Lọc từng vùng liên thông.
ROAD_MIN_REGION_FRAC=0.01      # vùng < 1% ảnh -> bỏ (small)
ROAD_REGION_MEAN_CONF=0.70     # P(road) trung bình của vùng < 0.70 -> bỏ (low_conf)
# (5) Kiểm tra cấp ảnh. Vi phạm -> road 'unreliable' (xử lý theo ROAD_UNRELIABLE_POLICY).
ROAD_MIN_TOTAL_FRAC=0.02       # road giữ lại < 2% ảnh: mảng road lẻ loi, không đáng tin (area_too_small)
ROAD_MAX_TOTAL_FRAC=0.60       # road > 60% ảnh: đáng ngờ với camera đường phố (area_too_large)
ROAD_MAX_REGIONS=4             # quá nhiều vùng rời (too_many_regions)
ROAD_MIN_LARGEST_SHARE=0.50    # khi có >= 2 vùng, vùng lớn nhất phải chiếm >= 50% road (fragmented)
ROAD_IMAGE_MEAN_CONF=0.75      # P(road) trung bình theo pixel (low_image_conf)
ROAD_MAX_UNCERTAIN_FRAC=0.10   # pixel argmax=road không phải flood/xe mà bị loại > 10% ảnh (uncertain_road)
# 'no_road' (giữ ảnh, không nhãn road) chỉ khi có bằng chứng thật là không thấy mặt đường. Nếu không giữ được vùng road
# nào nhưng argmax=road >= ROAD_NO_ROAD_MAX_RAW_FRAC ảnh, hoặc P(road) trung bình ở nửa dưới ảnh (trừ flood/xe)
# >= ROAD_NO_ROAD_MAX_LOWER_PROB, thì mặt đường có thể đang ở đó mà teacher không tách được -> 'road_missed' (unreliable).
ROAD_NO_ROAD_MAX_RAW_FRAC=0.01
ROAD_NO_ROAD_MAX_LOWER_PROB=0.15
ROAD_LOWER_START_FRAC=0.50     # 'nửa dưới' bắt đầu từ 50% chiều cao
# Vị trí hợp lý cho camera ở mặt đất/dashcam/CCTV: mặt đường nằm ở nửa dưới ảnh, không ở dải trên cùng
# (dải trên cùng thường là trời, mái nhà, tường). Đặt False nếu có nhiều ảnh drone/nhìn từ trên cao.
ROAD_SPATIAL_CHECKS=True
ROAD_MIN_CENTROID_Y=0.45       # trọng tâm road (theo chiều cao, 0=trên, 1=dưới) phải >= 0.45 (implausible_position)
ROAD_TOP_BAND_FRAC=0.30        # dải trên cùng = 30% chiều cao
ROAD_MAX_TOP_BAND_SHARE=0.20   # tối đa 20% pixel road nằm trong dải trên cùng (road_in_top_band)
# (6) Ảnh có road 'unreliable':
#   'drop_image'        : bỏ ảnh (mặc định). YOLO-seg không có vùng ignore, nên giữ ảnh mà bỏ nhãn road
#                         sẽ dạy model rằng mặt đường nhìn thấy được là background.
#   'keep_without_road' : giữ ảnh (không mất flood/xe) nhưng không có polygon road.
ROAD_UNRELIABLE_POLICY='drop_image'
# (7) Kiểm tra trên nhãn: polygon road rasterize (như Ultralytics) trùng flood > tỉ lệ này -> bỏ polygon.
ROAD_FLOOD_MAX_OVERLAP_FRAC=0.0

# ======================= FLOOD / BỐI CẢNH ĐƯỜNG PHỐ =======================
# Camera đường phố: bỏ hẳn các thư mục WaterDataset là sông/hồ/biển/cảng. Mask thật của chúng gán mọi mặt nước là
# water, nên nếu giữ, model học 'nước nào cũng là ngập'. So khớp tên thư mục (không phân biệt hoa thường, cho phép *).
WATER_EXCLUDE_DIRS=('ADE20K_water','river_segs','boston_harbor*','houston_buffalo*','lake_exp')
AGREEMENT_IOU=0.5              # IoU WaterNet vs SegFormer water tối thiểu cho ảnh không có mask thật
MIN_REGION_FRAC=0.005          # vùng flood / xe < 0.5% ảnh bị loại
# Bài toán là ngập đường phố: ảnh phải có bối cảnh đô thị (tỉ lệ pixel SegFormer thuộc các lớp dưới).
STREET_CONTEXT_CLASS_NAMES=('road','sidewalk','building','house','skyscraper','car','truck','bus','van','minibike',
                            'bicycle','streetlight','traffic light','signboard','pole','fence','railing','awning')
STREET_CONTEXT_MIN_FRAC=0.05   # < 5% ảnh là bối cảnh đường phố -> loại (no_street_context)
NATURAL_WATER_CLASS_NAMES=('river','lake','sea')
NATURAL_WATER_MAX_FRAC=0.50    # >= 50% vùng flood bị SegFormer gọi là sông/hồ/biển ...
STREET_CONTEXT_STRONG_FRAC=0.15  # ... mà bối cảnh đường phố < 15% -> loại (natural_water_body)
STREET_CONTEXT_FILTER_TEST=True  # áp dụng cả cho test WaterDataset (test để đánh giá ngập đường phố)

# ======================= ẢNH ÂM (không ngập) =======================
# Ảnh âm CHỈ lấy từ nguồn được tuyển chọn là không ngập (thư mục có tên kiểu non_flooded/dry_road/...,
# hoặc khai báo trong NEGATIVE_INPUT_DIRS). Không bao giờ tạo ảnh âm bằng cách xoá flood mà teacher thấy.
NEGATIVE_INPUT_DIRS=[]         # glob tương đối với /kaggle/input, ví dụ ['my-dry-roads/*', 'rain-streets/wet']
NEGATIVE_AUTO_DISCOVER=True    # tự tìm thư mục tên non-flood/dry-road/wet-road/reflection/puddle/negative
NEGATIVE_DIR_NAMES={'negative','negatives','nonflood','nonflooded','nonflooding','noflood','noflooding','notflood',
                    'notflooded','dryroad','dryroads','drystreet','drystreets','wetroad','wetroads','wetstreet',
                    'wetstreets','normalroad','normalstreet','reflection','reflections','puddle','puddles','smallpuddle'}
NEGATIVE_CATEGORIES=('dry_road','wet_road','reflection','small_puddle')   # loại ảnh âm cần có
# Ảnh từ nguồn âm mà cả hai teacher cùng thấy nước lớn (IoU >= AGREEMENT_IOU và WaterNet >= 5% ảnh) có thể là
# ảnh ngập bị xếp nhầm hoặc sông/hồ -> loại. Nước nhỏ hơn (vũng, phản chiếu) vẫn giữ, chỉ ghi cảnh báo.
NEG_REJECT_WATER_FRAC=0.05
NEG_WARN_WATER_FRAC=0.01
# Ảnh thuộc nguồn ngập mà cả hai teacher không thấy nước: mặc định LOẠI (nhiều khả năng teacher bỏ sót).
ALLOW_TEACHER_NEGATIVES_FROM_FLOOD_SOURCES=False
# Tỉ lệ ảnh KHÔNG có flood (ảnh âm + ảnh chỉ có road/xe) trong mỗi split train/val.
NONFLOOD_MIN_FRAC=0.10         # dưới mức này -> dataset chưa sẵn sàng, báo thiếu bao nhiêu ảnh
NONFLOOD_TARGET_FRAC=0.15      # thừa thì cắt về mức này
NONFLOOD_MAX_FRAC=0.20         # không bao giờ vượt

# ======================= GROUP / SPLIT =======================
SEED=42
TEST_GROUP_FRACTION=0.15       # tỉ lệ group WaterDataset có mask thật giữ riêng làm test
VAL_FRACTION=0.15              # tỉ lệ ảnh train+val vào val (chia theo group)
MAX_PREFIX_GROUP_FRAC=0.20     # prefix tên file chiếm > 20% ảnh của một thư mục là prefix chung chung, không dùng làm group
GROUP_BY_NEAR_DUPLICATE=True   # gộp ảnh gần trùng (dHash) vào cùng group: frame cùng video không lọt sang split khác
DHASH_MAX_DISTANCE=6           # khoảng cách Hamming tối đa (trên 64 bit) để coi là gần trùng
# Thư mục con là VIDEO (cả thư mục = 1 group) khi >= VIDEO_MIN_ADJ_FRAC cặp ảnh liền kề (sắp theo tên) có dHash
# cách nhau <= VIDEO_ADJ_MAX_DISTANCE. Ngược lại là BỘ ẢNH RỜI (ADE20K_water, flooding_pixabay...): mỗi ảnh 1 group,
# chỉ ảnh gần trùng mới bị gộp. Trước đây cả bộ ảnh rời là 1 group và một cặp gần trùng giữa hai thư mục gộp
# luôn hai thư mục -> WaterDataset chỉ còn 1 group, không tách được test.
VIDEO_ADJ_MAX_DISTANCE=10
VIDEO_MIN_ADJ_FRAC=0.5
SPLIT_SEARCH_TRIALS=3000
SPLIT_MIN_GROUPS_FOR_CLASS=2   # class có mặt ở >= 2 group thì bắt buộc có trong cả train và val

# ======================= XE =======================
CAR_CONF=0.5                   # confidence tối thiểu của best_car_YOLOv11 (level 0-4)
CAR_IOU=0.5                    # IoU tối thiểu giữa box level và box mask yolo11x-seg
VEHICLE_MASK_CONF=0.25         # confidence của yolo11x-seg cho mask xe/xe máy/bus/truck
VEHICLE_COCO_NAMES=('car','motorcycle','bus','truck')  # mask dùng để trừ khỏi road; chỉ 'car' dùng để ghép level

# ======================= CHẤT LƯỢNG / PREVIEW =======================
QUALITY_MIN_IMAGES={'train':{'flood':20,'road':20},'val':{'flood':5,'road':5}}
QUALITY_MIN_TRAIN_CAR_INSTANCES=10  # dưới mức này chỉ cảnh báo; 0 trong train -> chưa sẵn sàng
QUALITY_MIN_TEST_IMAGES=20
ROAD_MAX_UNRELIABLE_RATE=0.50  # > 50% ảnh bị road unreliable -> cảnh báo (ngưỡng/teacher không hợp dữ liệu)
PREVIEW_PER_CATEGORY=8
PANEL_W=480

assert ROAD_UNRELIABLE_POLICY in ('drop_image','keep_without_road')
assert ROAD_FLOOD_GAP_PX>2*REGION_POLY_EPS_PX+1, 'ROAD_FLOOD_GAP_PX quá nhỏ so với REGION_POLY_EPS_PX'
assert 0<=NONFLOOD_MIN_FRAC<=NONFLOOD_TARGET_FRAC<=NONFLOOD_MAX_FRAC<=0.20, 'tỉ lệ ảnh không flood phải trong khoảng 10-20% và không vượt 20%'
print('[CONFIG]',{k:v for k,v in globals().items() if k.isupper() and not k.startswith('_') and k not in ('LIMIT','INPUT_ROOT','IMAGE_EXTS','MASK_EXTS','MASK_HINTS','WATER_WEIGHTS','CAR_LEVEL_WEIGHTS','WATER_ROOT','OLD_ROOT','ROADWAY_ROOT')})


## 1. Cài thư viện teacher

In [ ]:
%pip -q install segmentation-models-pytorch==0.2.0 transformers ultralytics


## 2. Tìm ảnh WaterDataset có ground-truth mask, ảnh nguồn cũ, Roadway và nguồn ảnh không ngập

Mask được nhận diện theo tên **thư mục** (`mask`, `groundtruth`, `annotations`, `gt`…) và/hoặc hậu tố filename. Nếu không tìm ra cặp ảnh/mask thì notebook dừng. Nguồn ảnh âm được tìm theo tên thư mục (`non_flooded`, `dry_road`, `wet_road`, `reflection`, `puddle`, `negative`…) hoặc `NEGATIVE_INPUT_DIRS`; nếu không có, notebook in rõ cần bổ sung gì.

Thư mục WaterDataset trong `WATER_EXCLUDE_DIRS` (mặc định `ADE20K_water`, `river_segs`, `boston_harbor*`, `houston_buffalo*`, `lake_exp`: sông/hồ/biển/cảng) bị bỏ hẳn, vì bài toán là camera đường phố và mask thật của chúng gán mọi mặt nước là water.


In [ ]:
import numpy as np
import cv2
from PIL import Image
IMAGE_EXTS={'.jpg','.jpeg','.png','.bmp','.webp'}
MASK_EXTS=IMAGE_EXTS|{'.tif','.tiff','.npy'}
MASK_DIR_TOKENS=('mask','masks','groundtruth','groundtruths','annotation','annotations','label','labels','gt')
MASK_SUFFIXES=('_mask','-mask','_label','-label','_gt','-gt','_water','-water')
# Folder names that never identify a video/source (split folders, export folders). 'yolodataset' matters:
# the old dataset lives in yolo_dataset/images/{train,val}, which previously made all its images ONE group.
GENERIC_DIRS={'images','image','imgs','img','frames','frame','train','val','valid','validation','test','trainimages','valimages','validimages','testimages','waterdataset','waterdatasetzip',
              'jpegimages','yolodataset','dataset','datasets','data','all','allimages','photos','pictures','raw','flood','flooded','flooding','floods'}

def _is_mask_token(x):
    n=norm(x)
    # 'gt' only as a whole token/prefix/suffix: a plain substring test also matched 'night', 'length', 'light'.
    return n in {'gt','gts'} or n.startswith('gt') or n.endswith('gt') or any(t in n for t in MASK_DIR_TOKENS if t!='gt')
def is_mask_dir(p):
    # Only folder names decide; file names are handled by MASK_SUFFIXES.
    parts=p.parts[:-1] if Path(p).suffix.lower() in MASK_EXTS else p.parts
    return any(_is_mask_token(x) for x in parts)
def stem_key(p):
    s=p.stem.lower()
    for suffix in MASK_SUFFIXES:
        if s.endswith(suffix): s=s[:-len(suffix)]; break
    return s

def read_binary_mask(p):
    if p.suffix.lower()=='.npy': a=np.load(p)
    else: a=np.array(Image.open(p))
    if a.ndim==3: a=a[...,0]
    return a>0

water_all=[p for p in WATER_ROOT.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTS and not is_mask_dir(p) and not any(p.stem.lower().endswith(s) for s in MASK_SUFFIXES)]
import fnmatch
def water_excluded(p):
    return any(fnmatch.fnmatch(x.lower(),pat.lower()) for x in p.relative_to(WATER_ROOT).parts[:-1] for pat in WATER_EXCLUDE_DIRS)
excluded_counts=collections.Counter(str(p.parent.relative_to(WATER_ROOT)) for p in water_all if water_excluded(p))
water_all=[p for p in water_all if not water_excluded(p)]
print(f'[WATER_EXCLUDE_DIRS] bỏ {sum(excluded_counts.values())} ảnh sông/hồ/biển:',dict(excluded_counts))
water_masks=[p for p in WATER_ROOT.rglob('*') if p.is_file() and p.suffix.lower() in MASK_EXTS and is_mask_dir(p)]
# Also include sibling mask-named files (e.g. frame_mask.png beside frame.jpg).
water_masks += [p for p in WATER_ROOT.rglob('*') if p.is_file() and p.suffix.lower() in MASK_EXTS and any(p.stem.lower().endswith(s) for s in MASK_SUFFIXES)]
mask_by_stem=collections.defaultdict(list)
for p in water_masks: mask_by_stem[stem_key(p)].append(p)
water_pairs=[]
for im in water_all:
    candidates=mask_by_stem.get(im.stem.lower(),[])
    if not candidates: candidates=mask_by_stem.get(stem_key(im),[])
    if candidates:
        # Prefer masks nested in a mask/GT folder over incidental similarly named files.
        candidates=sorted(set(candidates),key=lambda p:(not is_mask_dir(p),len(p.parts)))
        water_pairs.append((im,candidates[0]))
if not water_pairs:
    mask_dirs=[str(p) for p in WATER_ROOT.rglob('*') if p.is_dir() and is_mask_dir(p)]
    raise FileNotFoundError('WaterDataset image/mask pairs not found. Mask-like folders: '+repr(mask_dirs[:30])+'; inspect the printed tree and update the mask-pair matcher.')

# Old dataset: deliberately skip every labels directory and every annotation/config file.
old_images=[]
for p in OLD_ROOT.rglob('*'):
    if not p.is_file() or p.suffix.lower() not in IMAGE_EXTS: continue
    rel_parts=[norm(x) for x in p.relative_to(OLD_ROOT).parts]
    if any(x in {'label','labels','annotation','annotations'} for x in rel_parts[:-1]): continue
    if any(x.endswith('yaml') or x.endswith('yml') for x in rel_parts): continue
    old_images.append(p)
if not old_images: raise FileNotFoundError('Không tìm thấy ảnh trong flood-yolo-seg-dataset sau khi bỏ qua labels/.')
print(f'WaterDataset: {len(water_all)} ảnh, {len(water_masks)} mask candidates, {len(water_pairs)} cặp ảnh/mask thật')
print(f'Old dataset: {len(old_images)} ảnh được lấy; nội dung labels/ và data.yaml không được mở.')

def scan_roadway(root):
    imgs=[p for p in root.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTS and not is_mask_dir(p) and not any(p.stem.lower().endswith(s) for s in MASK_SUFFIXES)]
    masks=[p for p in root.rglob('*') if p.is_file() and p.suffix.lower() in MASK_EXTS and (is_mask_dir(p) or any(p.stem.lower().endswith(s) for s in MASK_SUFFIXES))]
    lookup=collections.defaultdict(list)
    for p in masks: lookup[stem_key(p)].append(p)
    pairs=[]
    for im in imgs:
        ms=lookup.get(im.stem.lower(),[]) or lookup.get(stem_key(im),[])
        if ms: pairs.append((im,sorted(set(ms),key=lambda p:(not is_mask_dir(p),len(p.parts)))[0]))
    return imgs,masks,pairs

if ROADWAY_ROOT is not None:
    roadway_all,roadway_masks,roadway_pairs=scan_roadway(ROADWAY_ROOT)
    if not roadway_all: raise FileNotFoundError(f'Không tìm thấy ảnh trong {ROADWAY_ROOT}.')
    roadway_gt={im:mask for im,mask in roadway_pairs}
    print(f'Roadway Flooding: {len(roadway_all)} ảnh, {len(roadway_masks)} mask candidates, {len(roadway_pairs)} cặp khớp.')
    if roadway_pairs: print('Roadway mask files match image stems: dùng làm flood GT.')
    else: print('Không có mask khớp ảnh trong Roadway Flooding: dùng ảnh này cho teacher pseudo-label.')
else:
    roadway_all=[]; roadway_masks=[]; roadway_pairs=[]; roadway_gt={}
    print('[INPUT MISSING] Roadway Flooding chưa có trong cây /kaggle/input; source sẽ được bỏ qua lần chạy này.')

# ---------- Negative (non-flood) sources ----------
# Only folders curated as "no flood" (by name or NEGATIVE_INPUT_DIRS). Never WaterDataset (all water scenes).
def is_negative_dir_name(name):
    n=norm(name)
    return n in NEGATIVE_DIR_NAMES or (n.startswith(('non','no','not','without')) and 'flood' in n)
def negative_category(path):
    # Match whole words (prefix of a token), not substrings: 'flood_train_001' must not hit 'rain'.
    toks=[t for x in path.parts for t in re.split(r'[^a-z]+',x.lower()) if t]
    has=lambda *keys: any(t.startswith(k) for t in toks for k in keys)
    if has('reflect','mirror','glass'): return 'reflection'
    if has('puddle'): return 'small_puddle'
    if has('wet','rain'): return 'wet_road'
    if has('dry'): return 'dry_road'
    return 'unspecified'
neg_dirs=[p for pattern in NEGATIVE_INPUT_DIRS for p in INPUT_ROOT.glob(pattern) if p.is_dir()]
if NEGATIVE_AUTO_DISCOVER: neg_dirs+=[p for p in INPUT_ROOT.rglob('*') if p.is_dir() and is_negative_dir_name(p.name)]
neg_dirs=sorted({p for p in neg_dirs if p!=WATER_ROOT and WATER_ROOT not in p.parents and not is_mask_dir(p)})
neg_dirs=[p for p in neg_dirs if not any(q in p.parents for q in neg_dirs)]   # keep top-most folders only
negative_images=[]; negative_root={}
for d in neg_dirs:
    for p in d.rglob('*'):
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS and not is_mask_dir(p.relative_to(d)) and p not in negative_root:
            negative_images.append(p); negative_root[p]=d
# An image inside a negative folder is never used as a flood-source image.
neg_set=set(negative_images)
old_images=[p for p in old_images if p not in neg_set]
roadway_all=[p for p in roadway_all if p not in neg_set]; roadway_gt={k:v for k,v in roadway_gt.items() if k not in neg_set}
neg_cat_counts=collections.Counter(negative_category(p) for p in negative_images)
if negative_images:
    print(f'[NEGATIVE] {len(negative_images)} ảnh không ngập từ {len(neg_dirs)} thư mục:',*[str(d) for d in neg_dirs],sep='\n  ')
    print('[NEGATIVE] theo loại:',dict(neg_cat_counts))
else:
    print('[NEGATIVE MISSING] Không có nguồn ảnh không ngập trong /kaggle/input. Dataset sẽ KHÔNG có ảnh âm có nhãn rỗng.\n'
          '  Cần add Kaggle Dataset chứa ảnh đường phố không ngập: đường khô, đường ướt sau mưa (không ngập), phản chiếu trên\n'
          '  kính/gương/mặt đường, vũng nước nhỏ. Đặt ảnh trong thư mục tên như non_flooded/, dry_road/, wet_road/,\n'
          '  reflection/, puddle/ (tự nhận), hoặc khai báo glob trong NEGATIVE_INPUT_DIRS ở cell cấu hình.')
missing_neg_cats=[c for c in NEGATIVE_CATEGORIES if neg_cat_counts.get(c,0)==0]
if negative_images and missing_neg_cats:
    print('[NEGATIVE] Chưa có (theo tên thư mục) loại:',missing_neg_cats,'- ảnh loại unspecified không được tính vào các loại này.')


## 3. Group theo video/nguồn và giữ test ground-truth riêng

Group = thư mục con có nghĩa **nếu nó là video** (các ảnh liền kề theo tên gần giống nhau, xem `VIDEO_MIN_ADJ_FRAC`). Thư mục con là bộ ảnh rời (vd. `ADE20K_water`, `flooding_pixabay`) thì mỗi ảnh là một group. Trong thư mục phẳng hoặc bộ ảnh rời, prefix tên file chung (vd. `clip07_frame_0012` → `clip07`) là một group, trừ khi prefix quá chung; còn lại mỗi file một group. Sau đó ảnh gần trùng (dHash) được gộp vào cùng group để frame cùng video không lọt sang split khác; vì bộ ảnh rời đã tách từng ảnh, một cặp gần trùng giữa hai thư mục chỉ gộp đúng các ảnh đó chứ không gộp cả hai thư mục. Bản trước gom cả dataset cũ (thư mục phẳng) thành **một** group nên toàn bộ road/car rơi vào val.

Test gồm 15% group WaterDataset có mask thật; toàn bộ group đó bị loại khỏi train/val.


In [ ]:
random.seed(SEED)
# ---------- Group = video / source, so frames of one video never end up in two splits ----------
# 1) a meaningful sub-folder (not images/train/val/yolo_dataset/...) is one group IF it is a video/time-lapse
#    (consecutive files look alike); a photo collection sub-folder (ADE20K_water, flooding_pixabay) is treated like a flat folder;
# 2) in a flat folder / photo collection, a file-name prefix shared by several files ("clip07_frame_0012.jpg" -> "clip07",
#    Roboflow "_jpg.rf.<hash>" removed) is one group, unless the prefix is generic (> MAX_PREFIX_GROUP_FRAC of the folder);
# 3) otherwise one file = one group; then near-duplicate images (dHash) are merged into the same group.
# The previous version put every image of a flat folder into ONE group ('old/unstructured-source'), so the whole
# old dataset (the only source of road/car) landed in val.
GENERIC_PREFIXES={'image','img','im','frame','photo','pic','picture','screenshot','download','images','flood','flooded','file','scene'}
def filename_prefix(path):
    s=path.stem.lower()
    s=re.sub(r'[_.-](jpe?g|png|bmp|webp)\.rf\.[0-9a-f]+$','',s)          # Roboflow export suffix
    s=re.sub(r'(?:[\s_.\-()]*(?:frame|img|image|im|f)?[\s_.\-()]*\d+[\s_.\-()]*)+$','',s)  # trailing frame numbers
    s=s.strip(' _.-()')
    return s if len(s)>=2 and s not in GENERIC_PREFIXES else None

def folder_is_video(files):
    # Video / time-lapse: consecutive frames (sorted by name) look alike. Photo collection: they do not.
    if len(files)<3: return True
    hs=[dhash(p) for p in sorted(files)]
    pairs=[(a,b) for a,b in zip(hs,hs[1:]) if a is not None and b is not None]
    if not pairs: return True
    close=sum(int(POPCOUNT8[a^b].sum())<=VIDEO_ADJ_MAX_DISTANCE for a,b in pairs)
    return close/len(pairs)>=VIDEO_MIN_ADJ_FRAC

def base_groups(paths,root_of,tag):
    info={}
    for p in paths:
        root=root_of(p); rel=p.relative_to(root)
        # Category folders (dry_road/, non_flooded/, flooded/...) describe content, not a video -> not a group.
        parts=[x for x in rel.parts[:-1] if norm(x) not in GENERIC_DIRS and not is_mask_dir(Path(x)) and not is_negative_dir_name(x)]
        info[p]=('/'.join(parts),filename_prefix(p),str(root))
    files_by_dir=collections.defaultdict(list)
    for p,(d,_,r) in info.items():
        if d: files_by_dir[(r,d)].append(p)
    video_dirs={k for k,fs in files_by_dir.items() if folder_is_video(fs)}
    collections_=sorted(d for (r,d) in files_by_dir if (r,d) not in video_dirs)
    if files_by_dir:
        print(f'  {tag}: {len(video_dirs)} thư mục video (1 group/thư mục), {len(collections_)} bộ ảnh rời (group theo prefix/ảnh):',
              collections_[:10])
    dir_count=collections.Counter((r,d) for d,_,r in info.values())
    pre_count=collections.Counter((r,d,pre) for d,pre,r in info.values() if pre)
    out={}
    for p,(d,pre,r) in info.items():
        if d and (r,d) in video_dirs: out[p]=f'{tag}/{d}'
        elif pre and 2<=pre_count[(r,d,pre)]<=max(2,MAX_PREFIX_GROUP_FRAC*dir_count[(r,d)]): out[p]=f'{tag}/{d+"/" if d else ""}prefix:{pre}'
        else: out[p]=f'{tag}/file:{p.relative_to(r).as_posix()}'
    return out

POPCOUNT8=np.array([bin(i).count('1') for i in range(256)],np.uint8)
_DHASH_CACHE={}
def dhash(p):
    if p not in _DHASH_CACHE: _DHASH_CACHE[p]=_dhash(p)
    return _DHASH_CACHE[p]
def _dhash(p):
    try:
        im=Image.open(p); im.draft('L',(128,128))
        a=np.asarray(im.convert('L').resize((9,8),Image.BILINEAR),np.int16)
    except Exception: return None
    return np.packbits((a[:,1:]>a[:,:-1]).ravel())   # 64-bit difference hash as 8 bytes

def merge_near_duplicates(paths,groups,block=256):
    # Union groups that contain near-identical images (Hamming distance <= DHASH_MAX_DISTANCE).
    t0=time.perf_counter()
    names=sorted(set(groups.values())); gid={n:i for i,n in enumerate(names)}; parent=list(range(len(names)))
    def find(i):
        while parent[i]!=i: parent[i]=parent[parent[i]]; i=parent[i]
        return i
    hashes=[(p,dhash(p)) for p in paths]
    hashes=[(p,h) for p,h in hashes if h is not None and 4<=int(POPCOUNT8[h].sum())<=60]   # skip flat/blank images
    if len(hashes)>1:
        H=np.stack([h for _,h in hashes]); G=np.array([gid[groups[p]] for p,_ in hashes])
        for s in range(0,len(H),block):
            d=POPCOUNT8[H[s:s+block,None,:]^H[None,:,:]].sum(-1,dtype=np.uint16)
            ii,jj=np.nonzero(d<=DHASH_MAX_DISTANCE); gi=G[ii+s]; gj=G[jj]; keep=gi!=gj
            for a,b in set(zip(gi[keep].tolist(),gj[keep].tolist())):
                ra,rb=find(a),find(b)
                if ra!=rb: parent[max(ra,rb)]=min(ra,rb)
    merged={p:names[find(gid[g])] for p,g in groups.items()}
    n_before=len(names); n_after=len(set(merged.values()))
    print(f'  near-duplicate merge: {n_before} -> {n_after} groups ({time.perf_counter()-t0:.1f}s, {len(hashes)} hashed)')
    return merged

def build_groups(paths,root_of,tag):
    if not paths: return {}
    g=base_groups(paths,root_of,tag)
    if GROUP_BY_NEAR_DUPLICATE: g=merge_near_duplicates(paths,g)
    sizes=collections.Counter(g.values())
    print(f'[GROUPS] {tag}: {len(paths)} ảnh, {len(sizes)} group, lớn nhất {sizes.most_common(1)[0][1]} ảnh',
          '| top:',sizes.most_common(3))
    return g

water_group=build_groups(water_all,lambda p:WATER_ROOT,'water')
old_group=build_groups(old_images,lambda p:OLD_ROOT,'old')
roadway_group=build_groups(roadway_all,lambda p:ROADWAY_ROOT,'roadway') if roadway_all else {}
negative_group=build_groups(negative_images,lambda p:negative_root[p],'negative')

# ---------- Test = held-out WaterDataset groups with real masks ----------
water_groups=collections.defaultdict(list)
for im,mask in water_pairs: water_groups[water_group[im]].append((im,mask))
if len(water_groups)<2:
    raise ValueError(f'WaterDataset có {len(water_groups)} group mask thật; cần ít nhất 2 group để giữ test riêng theo video/nguồn. '
                     f'Xem dòng [GROUPS] water ở trên; thử tăng VIDEO_MIN_ADJ_FRAC hoặc giảm DHASH_MAX_DISTANCE.')
all_water_group_names=sorted(water_groups)
rng=random.Random(SEED); rng.shuffle(all_water_group_names)
n_test=max(1,round(len(all_water_group_names)*TEST_GROUP_FRACTION))
test_groups=set(all_water_group_names[:n_test])
print(f'{len(water_groups)} GT groups; test giữ {len(test_groups)} groups, {sum(len(water_groups[g]) for g in test_groups)} ảnh.')

def rec(image,source,group,gt_mask=None,split=None,neg_category=None):
    return {'image':image,'gt_mask':gt_mask,'source':source,'group':group,'split':split,'neg_category':neg_category}
records=[]
for group,items in water_groups.items():
    for im,mask in items: records.append(rec(im,'water',group,mask,'test' if group in test_groups else None))
paired_images={im for im,_ in water_pairs}
for im in water_all:
    if im in paired_images: continue
    if 'test_images_from_other_papers' in [x.lower() for x in im.parts]: continue
    if water_group[im] in test_groups: continue   # whole test group is excluded from train/val
    records.append(rec(im,'water',water_group[im]))
for im in old_images: records.append(rec(im,'old',old_group[im]))
for im in roadway_all: records.append(rec(im,'roadway',roadway_group[im],roadway_gt.get(im)))
for im in negative_images: records.append(rec(im,'negative',negative_group[im],neg_category=negative_category(im)))

full_source_counts=dict(collections.Counter(r['source'] for r in records))
def choose_group_balanced(items,cap,rng):
    if cap is None or len(items)<=cap: return list(items)
    buckets=collections.defaultdict(list)
    for item in items: buckets[item['group']].append(item)
    keys=list(buckets); rng.shuffle(keys)
    for key in keys: rng.shuffle(buckets[key])
    chosen=[]
    while len(chosen)<cap and keys:
        next_keys=[]
        for key in keys:
            if buckets[key] and len(chosen)<cap:
                chosen.append(buckets[key].pop())
            if buckets[key]: next_keys.append(key)
        keys=next_keys
    return chosen

if LIMIT is not None:
    limited=[]
    for source in sorted({r['source'] for r in records}):
        source_items=[r for r in records if r['source']==source]
        rng=random.Random(SEED+sum(map(ord,source)))
        # Reserve a representative test sample from the GT WaterDataset before sampling the rest.
        test_items=[r for r in source_items if r['split']=='test']
        other_items=[r for r in source_items if r['split']!='test']
        reserve=min(len(test_items),max(1,round(LIMIT*TEST_GROUP_FRACTION))) if test_items else 0
        selected=choose_group_balanced(test_items,reserve,rng)
        selected+=choose_group_balanced(other_items,max(0,LIMIT-len(selected)),rng)
        limited.extend(selected)
        print(f'[LIMIT] {source}: {len(source_items)} -> {len(selected)} images (limit {LIMIT})')
    records=limited
print('Candidates after LIMIT:',len(records),'| by source:',dict(collections.Counter(r['source'] for r in records)),
      '| true-mask:',sum(r['gt_mask'] is not None for r in records),'| groups:',len({r['group'] for r in records}))


## 4. Clone V-FloodNet và load đúng WaterNet với `strict=True`

Repo lưu `link_efficientb4_model.pth` như model PyTorch. Theo `train_image_seg.py` của repo, kiến trúc là `smp.Linknet(encoder_name="efficientnet-b4", encoder_depth=5, encoder_weights="imagenet", in_channels=3, classes=1, activation="sigmoid")`. Load model pickle của repo để lấy state, dựng đúng định nghĩa LinkNet từ repo, rồi `load_state_dict(..., strict=True)`. Bất kỳ lỗi nào dừng notebook.

In [ ]:
import subprocess, sys
from pathlib import Path
VF_REPO=Path('/kaggle/working/V-FloodNet')
if not VF_REPO.exists(): subprocess.run(['git','clone','--depth','1','https://github.com/xmlyqing00/V-FloodNet.git',str(VF_REPO)],check=True)
sys.path.insert(0,str(VF_REPO))
import torch
import segmentation_models_pytorch as smp
from test_image_seg import predict_pil

try:
    load_t0=time.perf_counter(); print(f'[LOAD] WaterNet checkpoint: {WATER_WEIGHTS}')
    serialized=torch.load(WATER_WEIGHTS,map_location='cpu',weights_only=False)
    if isinstance(serialized,torch.nn.Module): state=serialized.state_dict()
    elif isinstance(serialized,dict):
        state=serialized.get('state_dict',serialized.get('weights',serialized))
    else: raise TypeError(f'Unexpected WaterNet checkpoint type: {type(serialized)}')
    water_model=smp.Linknet(
        encoder_name='efficientnet-b4', encoder_depth=5, encoder_weights=None,
        in_channels=3, classes=1, activation='sigmoid')
    water_model.load_state_dict(state,strict=True)
    water_model.eval().to('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'[LOAD OK] WaterNet architecture=strict Linknet EfficientNet-B4, {time.perf_counter()-load_t0:.1f}s')
except Exception as e:
    print(f'[LOAD ERROR] WaterNet: {type(e).__name__}: {e}')
    raise RuntimeError('WaterNet không load được với kiến trúc chính thức Linknet/EfficientNet-B4 và strict=True; dừng để không tạo nhãn sai.') from e
print('WaterNet loaded strict=True:',type(water_model).__name__)

## 5. Load SegFormer ADE20K và hai teacher ô tô

In [ ]:
from transformers import AutoImageProcessor, SegformerForSemanticSegmentation
from ultralytics import YOLO
from torchvision.ops import box_iou
import numpy as np, cv2, re, collections

DEVICE=0 if torch.cuda.is_available() else 'cpu'
seg_name=SEGFORMER_MODEL
load_t0=time.perf_counter(); print(f'[LOAD] SegFormer ADE20K: {seg_name}')
try:
    seg_processor=AutoImageProcessor.from_pretrained(seg_name)
    seg_model=SegformerForSemanticSegmentation.from_pretrained(seg_name).eval()
    if torch.cuda.is_available(): seg_model=seg_model.cuda()
    print(f'[LOAD OK] SegFormer ADE20K in {time.perf_counter()-load_t0:.1f}s')
except Exception as e:
    print(f'[LOAD ERROR] SegFormer ADE20K: {type(e).__name__}: {e}')
    raise
label_by_id={int(i):str(name).strip().lower() for i,name in seg_model.config.id2label.items()}
road_ids={i for i,n in label_by_id.items() if n in {x.lower() for x in ROAD_CLASS_NAMES}}
water_names={'water','sea','river','lake'}
water_class_ids={i for i,n in label_by_id.items() if n in water_names}
if len(road_ids)!=len(set(ROAD_CLASS_NAMES)): raise ValueError(f'ROAD_CLASS_NAMES={ROAD_CLASS_NAMES} không khớp hết config.id2label; tìm thấy {road_ids}: {label_by_id}')
if not water_class_ids: raise ValueError(f'SegFormer ADE20K không có water/sea/river/lake trong config.id2label: {label_by_id}')
print('ADE20K road ids:',{i:label_by_id[i] for i in road_ids},'water ids:',{i:label_by_id[i] for i in water_class_ids})
def ids_for(names,what,required=True):
    wanted={x.lower() for x in names}; ids={i for i,n in label_by_id.items() if n in wanted}
    missing=wanted-{label_by_id[i] for i in ids}
    if missing: print(f'[WARN] {what}: không có trong id2label -> bỏ qua {sorted(missing)}')
    if required and not ids: raise ValueError(f'{what}: không tìm thấy lớp nào trong id2label')
    return ids
context_ids=ids_for(STREET_CONTEXT_CLASS_NAMES,'STREET_CONTEXT_CLASS_NAMES')
natural_water_ids=ids_for(NATURAL_WATER_CLASS_NAMES,'NATURAL_WATER_CLASS_NAMES')
print('ADE20K street-context:',{i:label_by_id[i] for i in sorted(context_ids)})
print('ADE20K natural water:',{i:label_by_id[i] for i in sorted(natural_water_ids)})
load_t0=time.perf_counter(); print(f'[LOAD] Car-level YOLO: {CAR_LEVEL_WEIGHTS}')
try:
    car_level_model=YOLO(str(CAR_LEVEL_WEIGHTS))
    print(f'[LOAD OK] Car-level YOLO in {time.perf_counter()-load_t0:.1f}s; classes={car_level_model.names}')
except Exception as e:
    print(f'[LOAD ERROR] Car-level YOLO: {type(e).__name__}: {e}'); raise
load_t0=time.perf_counter(); print('[LOAD] YOLO11x-seg COCO weights (download on first run if not cached)')
try:
    car_segmenter=YOLO('yolo11x-seg.pt')
    print(f'[LOAD OK] YOLO11x-seg in {time.perf_counter()-load_t0:.1f}s')
except Exception as e:
    print(f'[LOAD ERROR] YOLO11x-seg: {type(e).__name__}: {e}'); raise
# COCO ids by name from the segmenter itself (not hard-coded).
coco_by_name={str(n).lower():int(i) for i,n in car_segmenter.names.items()}
missing=[n for n in VEHICLE_COCO_NAMES if n not in coco_by_name]
if 'car' not in coco_by_name: raise ValueError(f'yolo11x-seg không có lớp car: {car_segmenter.names}')
if missing: print('[WARN] yolo11x-seg thiếu lớp xe:',missing)
VEHICLE_COCO_IDS=sorted(coco_by_name[n] for n in VEHICLE_COCO_NAMES if n in coco_by_name)
COCO_CAR_ID=coco_by_name['car']
print('COCO vehicle ids:',{i:car_segmenter.names[i] for i in VEHICLE_COCO_IDS},'| car id:',COCO_CAR_ID)
print('Car-level classes:',car_level_model.names)


## 6. Teacher inference, flood, bối cảnh đường phố, xe, road và polygon labels

Với mỗi ảnh (`process_image`):
1. **Flood**: mask thật nếu có; ảnh nguồn âm → nhãn flood rỗng theo nguồn (teacher vẫn chạy để ghi điểm; nếu cả hai teacher cùng thấy nước lớn thì loại ảnh); còn lại WaterNet khi IoU với SegFormer water ≥ `AGREEMENT_IOU`. Ảnh nguồn ngập mà teacher không thấy nước bị loại (không biến thành ảnh âm).
2. **Bối cảnh**: loại ảnh có < `STREET_CONTEXT_MIN_FRAC` pixel đường phố (`no_street_context`) và ảnh mà vùng flood chủ yếu là sông/hồ/biển (`natural_water_body`).
3. **Xe**: level (conf ≥ 0.5) + mask car ghép IoU ≥ 0.5; mask mọi xe (car/motorcycle/bus/truck) được trừ khỏi road.
4. **Road** = road SegFormer (P(road) ≥ ngưỡng, không lẫn nước) − flood − xe, lọc vùng nhỏ/confidence thấp; kiểm tra cấp ảnh: quá nhỏ, quá lớn, quá nhiều vùng, vụn, confidence thấp, vị trí không hợp lý (trọng tâm quá cao, nằm ở dải trên cùng), còn nhiều road không chắc chắn, hoặc không giữ được vùng nào dù có dấu vết mặt đường (`road_missed`: argmax=road ≥ 1% ảnh hoặc P(road) nửa dưới ≥ 0.15) → `unreliable` → loại ảnh (`ROAD_UNRELIABLE_POLICY`).
5. **Test**: chỉ ghi flood GT; road/car pseudo không được ghi.

Preview: mọi ảnh có road → `preview_road/`, mọi ảnh road unreliable → `preview_road_rejected/`. Mỗi panel gồm ảnh gốc, nhãn xuất ra, road mask (xám = vùng bị loại, xanh = xe) và flood mask (so với SegFormer/WaterNet để soi lỗi biên). Polygon đẹp không phải bằng chứng nhãn đúng.

**Hạn chế**: road chỉ đến từ SegFormer ADE20K (mặc định B5, `SEGFORMER_MODEL`), không được train cho cảnh ngập, confidence chưa calibrate; kiểm tra vị trí giả định camera ở mặt đất (tắt bằng `ROAD_SPATIAL_CHECKS=False` nếu có nhiều ảnh drone).


In [ ]:
import textwrap
WORK=Path('/kaggle/working')
OUT=WORK/'dataset'; PREVIEW=WORK/'preview'; PREVIEW_ROAD=WORK/'preview_road'; PREVIEW_ROAD_REJECTED=WORK/'preview_road_rejected'
for d in (OUT,PREVIEW,PREVIEW_ROAD,PREVIEW_ROAD_REJECTED):
    if d.exists(): shutil.rmtree(d)   # outputs of an earlier run must not mix with this run
    d.mkdir(parents=True)
for split in ('train','val','test'):
    (OUT/'images'/split).mkdir(parents=True,exist_ok=True)
    (OUT/'labels'/split).mkdir(parents=True,exist_ok=True)
CLASS_NAMES=['flood','road','car_L0','car_L1','car_L2','car_L3','car_L4']
CAR_CLASS_OFFSET=2
TORCH_DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def level_index(name):
    m=re.search(r'(?:level|l)[ _-]?([0-4])\s*$',str(name).lower())
    return int(m.group(1)) if m else None
car_level_map={int(i):level_index(n) for i,n in car_level_model.names.items()}
print('Car-level class -> level:',{car_level_model.names[i]:lv for i,lv in car_level_map.items()})
if any(lv is None for lv in car_level_map.values()): print('[WARN] Có lớp của best_car_YOLOv11 không đọc được level 0-4; detection của lớp đó bị bỏ.')
missing_levels=[k for k in range(5) if k not in car_level_map.values()]
if missing_levels: print(f'[WARN] best_car_YOLOv11 không có lớp cho level {missing_levels}: không thể có nhãn car_L{missing_levels}.')

def waternet_mask(pil):
    # Uses the official repo's preprocessing/inference/postprocessing function.
    pred=np.array(predict_pil(water_model,pil,model_dims=(416,416),device=TORCH_DEVICE))>0
    if pred.ndim==3: pred=pred[...,0]
    return cv2.resize(pred.astype(np.uint8),pil.size,interpolation=cv2.INTER_NEAREST).astype(bool)

def segformer_maps(pil):
    # Full-resolution ADE20K argmax ids plus P(road) and P(water-like) (softmax summed over the class ids).
    inputs=seg_processor(images=pil.convert('RGB'),return_tensors='pt')
    dev=next(seg_model.parameters()).device
    inputs={k:v.to(dev) for k,v in inputs.items()}
    with torch.inference_mode():
        logits=seg_model(**inputs).logits
        probs=logits.float().softmax(1)
        pr=torch.stack([probs[:,sorted(road_ids)].sum(1),probs[:,sorted(water_class_ids)].sum(1)],1)
        pr=torch.nn.functional.interpolate(pr,size=(pil.height,pil.width),mode='bilinear',align_corners=False)[0].clamp_(0,1).cpu().numpy()
        logits=torch.nn.functional.interpolate(logits,size=(pil.height,pil.width),mode='bilinear',align_corners=False)
        ids=logits.argmax(1)[0].cpu().numpy()
    return ids,pr[0],pr[1]

def contours_polygons(mask):
    # Car instances (unchanged behaviour).
    mask=np.asarray(mask,dtype=bool)
    h,w=mask.shape; min_area=MIN_REGION_FRAC*h*w
    contours,_=cv2.findContours(mask.astype(np.uint8),cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    rows=[]; accepted=[]
    for contour in contours:
        area=cv2.contourArea(contour)
        if area<min_area: continue
        poly=cv2.approxPolyDP(contour,0.002*cv2.arcLength(contour,True),True).reshape(-1,2)
        if len(poly)<3: continue
        rows.append(' '.join(f'{x/w:.6f} {y/h:.6f}' for x,y in poly))
        accepted.append((poly,area))
    return rows,accepted

# ---------- Region (flood/road) mask helpers ----------
def odd_ellipse(size_px):
    k=max(3,int(round(size_px))); k+=(k%2==0)
    return cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(k,k))

def dilate(mask,px):
    return cv2.dilate(np.asarray(mask,np.uint8),odd_ellipse(2*px+1)).astype(bool) if px>0 else np.asarray(mask,bool)

def _background_components(pad):
    # 4-connected background components of a zero-padded mask; the padding is one component (outside).
    n,lab,stats,_=cv2.connectedComponentsWithStats((~pad).astype(np.uint8),connectivity=4)
    return n,lab,stats,int(lab[0,0])

def fill_small_holes(mask,max_area):
    # Fill enclosed background holes whose area <= max_area (speckle inside a region).
    pad=np.pad(np.asarray(mask,bool),1,constant_values=False)
    n,lab,stats,outside=_background_components(pad)
    small=[i for i in range(1,n) if i!=outside and stats[i,cv2.CC_STAT_AREA]<=max_area]
    if small: pad=pad|np.isin(lab,small)
    return pad[1:-1,1:-1]

def bridge_holes(mask,slit=2,max_iter=50):
    # YOLO-seg polygons cannot hold holes; RETR_EXTERNAL would silently fill them. Cut a thin vertical slit
    # from each hole to the outside so the outer contour walks around the hole.
    pad=np.pad(np.asarray(mask,bool),1,constant_values=False).copy()
    for _ in range(max_iter):
        n,lab,stats,outside=_background_components(pad)
        holes=[i for i in range(1,n) if i!=outside]
        if not holes: break
        for i in holes:
            y0=int(stats[i,cv2.CC_STAT_TOP]); x0=int(np.flatnonzero(lab[y0]==i)[0]); y=y0-1
            while pad[y,x0]:  # padding row 0 is background, so the loop always stops
                pad[y,x0:x0+slit]=False; y-=1
    return pad[1:-1,1:-1]

def mask_to_polygons(mask,min_frac,eps_px=None):
    # Hole-aware polygons for region classes; returns integer pixel polygons (N,2).
    eps_px=REGION_POLY_EPS_PX if eps_px is None else eps_px
    mask=np.asarray(mask,bool); h,w=mask.shape; min_area=min_frac*h*w
    if not mask.any(): return []
    pad=np.pad(bridge_holes(mask),1,constant_values=False).astype(np.uint8)
    contours,_=cv2.findContours(pad,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_NONE)
    polys=[]
    for contour in contours:
        if cv2.contourArea(contour)<min_area: continue
        poly=cv2.approxPolyDP(contour,eps_px,True).reshape(-1,2)-1
        poly=np.clip(poly,0,[w-1,h-1])
        if len(poly)>=3: polys.append(poly)
    return polys

def poly_to_row(poly,w,h): return ' '.join(f'{x/w:.6f} {y/h:.6f}' for x,y in poly)

def rasterize(polys,h,w):
    # Same rasterizer Ultralytics uses for polygon labels (cv2.fillPoly, one polygon at a time).
    out=np.zeros((h,w),bool)
    for poly in polys:
        m=np.zeros((h,w),np.uint8); cv2.fillPoly(m,[np.asarray(poly,np.int32).reshape(-1,1,2)],1); out|=m.astype(bool)
    return out

def analyze_road(road_argmax,road_prob,water_prob,flood,flood_poly_mask,vehicle):
    """road = road_prediction - flood - vehicles, with pixel/region/image-level checks.
    Returns (polys, debug, masks). debug['status'] is 'ok', 'no_road' or 'unreliable'."""
    h,w=road_argmax.shape; hw=float(h*w)
    dbg={'raw_frac':float(road_argmax.mean())}
    water_ambig=road_argmax&(water_prob>ROAD_MAX_WATER_PROB)
    pred=road_argmax&(road_prob>=ROAD_PIXEL_CONF)&~water_ambig
    dbg['water_ambiguous_frac']=float(water_ambig.mean())
    dbg['pixel_conf_frac']=float(pred.mean())
    k=odd_ellipse(ROAD_MORPH_FRAC*min(h,w))
    pred=cv2.morphologyEx(pred.astype(np.uint8),cv2.MORPH_OPEN,k)
    pred=cv2.morphologyEx(pred,cv2.MORPH_CLOSE,k).astype(bool)
    pred=fill_small_holes(pred,REGION_FILL_HOLE_FRAC*hw)
    flood_ref=flood|flood_poly_mask
    blocked=dilate(flood_ref,ROAD_FLOOD_GAP_PX)|dilate(vehicle,ROAD_VEHICLE_GAP_PX)
    road=pred&~blocked                       # road = road_prediction - flood_mask - vehicles
    assert not (road&flood_ref).any(), 'road mask overlaps flood mask'
    n,lab,stats,_=cv2.connectedComponentsWithStats(road.astype(np.uint8),connectivity=8)
    conf_sum=np.bincount(lab.ravel(),weights=road_prob.ravel(),minlength=n)
    kept_ids=[]; region_reasons=collections.Counter(); regions=[]
    for i in range(1,n):
        area=int(stats[i,cv2.CC_STAT_AREA]); mean_conf=float(conf_sum[i]/area)
        reason='small' if area<ROAD_MIN_REGION_FRAC*hw else ('low_conf' if mean_conf<ROAD_REGION_MEAN_CONF else None)
        regions.append((round(area/hw,4),round(mean_conf,3),reason or 'kept'))
        if reason: region_reasons[reason]+=1
        else: kept_ids.append(i)
    regions.sort(key=lambda r:-r[0])
    kept=np.isin(lab,kept_ids) if kept_ids else np.zeros_like(road)
    rejected=road&~kept
    kept_areas=[int(stats[i,cv2.CC_STAT_AREA]) for i in kept_ids]
    ys=np.nonzero(kept)[0]
    dbg.update(n_regions=n-1,n_kept=len(kept_ids),n_rejected_small=region_reasons['small'],
               n_rejected_low_conf=region_reasons['low_conf'],regions=regions[:8],
               kept_frac=float(kept.mean()),kept_conf=float(road_prob[kept].mean()) if kept.any() else 0.0,
               largest_share=float(max(kept_areas)/sum(kept_areas)) if kept_areas else 0.0,
               centroid_y=float(ys.mean()/h) if len(ys) else 0.0,
               top_band_share=float((ys<ROAD_TOP_BAND_FRAC*h).mean()) if len(ys) else 0.0,
               uncertain_frac=float((road_argmax&~blocked&~kept).mean()))
    lower=~blocked; lower[:int(ROAD_LOWER_START_FRAC*h)]=False
    dbg['lower_road_prob']=float(road_prob[lower].mean()) if lower.any() else 0.0
    reasons=[]
    if kept.any():
        if dbg['kept_frac']<ROAD_MIN_TOTAL_FRAC: reasons.append('area_too_small')
        if dbg['kept_frac']>ROAD_MAX_TOTAL_FRAC: reasons.append('area_too_large')
        if dbg['n_kept']>ROAD_MAX_REGIONS: reasons.append('too_many_regions')
        if dbg['n_kept']>=2 and dbg['largest_share']<ROAD_MIN_LARGEST_SHARE: reasons.append('fragmented')
        if dbg['kept_conf']<ROAD_IMAGE_MEAN_CONF: reasons.append('low_image_conf')
        if ROAD_SPATIAL_CHECKS and dbg['centroid_y']<ROAD_MIN_CENTROID_Y: reasons.append('implausible_position')
        if ROAD_SPATIAL_CHECKS and dbg['top_band_share']>ROAD_MAX_TOP_BAND_SHARE: reasons.append('road_in_top_band')
    if dbg['uncertain_frac']>ROAD_MAX_UNCERTAIN_FRAC: reasons.append('uncertain_road')
    if not kept.any() and not reasons and (dbg['raw_frac']>=ROAD_NO_ROAD_MAX_RAW_FRAC or dbg['lower_road_prob']>=ROAD_NO_ROAD_MAX_LOWER_PROB):
        reasons.append('road_missed')   # road-like evidence but nothing reliable: an image without road label would teach 'road = background'
    dbg['status']='unreliable' if reasons else ('ok' if kept.any() else 'no_road')
    dbg['reasons']=reasons
    polys=[]; dropped_overlap=0; max_overlap=0
    if dbg['status']=='ok':
        for poly in mask_to_polygons(kept,ROAD_MIN_REGION_FRAC):
            r=rasterize([poly],h,w); ov=int((r&flood_ref).sum()); max_overlap=max(max_overlap,ov)
            if ov>ROAD_FLOOD_MAX_OVERLAP_FRAC*r.sum(): dropped_overlap+=1; continue
            polys.append(poly)
        if not polys: dbg['status']='no_road'
    dbg.update(n_polys=len(polys),n_polys_dropped_overlap=dropped_overlap,poly_flood_overlap_px_max=max_overlap)
    return polys,dbg,{'kept':kept,'rejected':rejected}

def car_teacher(image_path,w,h):
    """Level from best_car_YOLOv11 (conf >= CAR_CONF) + mask from yolo11x-seg 'car' (box IoU >= CAR_IOU).
    Unmatched detections are dropped. Also returns the union of all vehicle masks (subtracted from road)."""
    level=car_level_model.predict(str(image_path),conf=CAR_CONF,device=DEVICE,verbose=False)[0]
    seg=car_segmenter.predict(str(image_path),classes=VEHICLE_COCO_IDS,conf=VEHICLE_MASK_CONF,device=DEVICE,retina_masks=True,verbose=False)[0]
    vehicle=np.zeros((h,w),bool); masks=[]; car_idx=[]; sboxes=np.zeros((0,4),np.float32)
    if seg.masks is not None and seg.boxes is not None and len(seg.boxes):
        sboxes=seg.boxes.xyxy.cpu().numpy(); scls=seg.boxes.cls.cpu().numpy().astype(int)
        for i,m in enumerate(seg.masks.data.cpu().numpy()):
            fm=cv2.resize((m>0.5).astype(np.uint8),(w,h),interpolation=cv2.INTER_NEAREST).astype(bool)
            masks.append(fm); vehicle|=fm
            if scls[i]==COCO_CAR_ID: car_idx.append(i)
    rows=[]; dets=[]; used=set()
    if level.boxes is not None and len(level.boxes):
        for cls,box,conf in zip(level.boxes.cls.cpu().numpy(),level.boxes.xyxy.cpu().numpy(),level.boxes.conf.cpu().numpy()):
            if conf<CAR_CONF: continue
            d={'level':car_level_map.get(int(cls)),'conf':round(float(conf),3),'iou':0.0,'matched':False,'reason':''}
            dets.append(d)
            if d['level'] is None: d['reason']='unknown_level_name'; continue
            if not car_idx: d['reason']='no_car_mask'; continue
            ious=box_iou(torch.as_tensor(box).float().view(1,4),torch.as_tensor(sboxes[car_idx]).float())[0].numpy()
            order=np.argsort(ious)[::-1]; d['iou']=round(float(ious[order[0]]),3)
            match=next((car_idx[int(j)] for j in order if ious[j]>=CAR_IOU and car_idx[int(j)] not in used),None)
            if match is None: d['reason']=f'no_mask_iou>={CAR_IOU}'; continue
            used.add(match)
            x1,y1,x2,y2=box; dx=.03*(x2-x1); dy=.03*(y2-y1)
            roi=np.zeros((h,w),bool); roi[max(0,int(y1-dy)):min(h,int(y2+dy)),max(0,int(x1-dx)):min(w,int(x2+dx))]=True
            polys,_=contours_polygons(masks[match]&roi)
            if not polys: d['reason']='mask_below_min_area'; continue
            d['matched']=True; d['reason']='exported'
            rows.extend(f'{CAR_CLASS_OFFSET+d["level"]} {poly}' for poly in polys)
    return rows,vehicle,dets

def process_image(rec):
    """Runs every teacher on one record. Returns a result dict; 'status' is 'kept' or 'rejected'.
    'view' holds full-resolution arrays for previews only (dropped by the caller)."""
    src=rec['image']; pil=Image.open(src).convert('RGB'); w,h=pil.size
    r={'status':'kept','reasons':[],'warnings':[],'size':(w,h),'rows':[],'road':None,'dets':[],'scores':{}}
    S=r['scores']
    ids,road_prob,water_prob=segformer_maps(pil)
    seg_water=np.isin(ids,list(water_class_ids)); road_argmax=np.isin(ids,list(road_ids))
    S['segformer_water_frac']=float(seg_water.mean())
    S['street_context_frac']=float(np.isin(ids,list(context_ids)).mean())
    natural=np.isin(ids,list(natural_water_ids)); S['natural_water_image_frac']=float(natural.mean())
    wn=None; flood=np.zeros((h,w),bool)
    if rec['gt_mask'] is not None:
        flood=cv2.resize(read_binary_mask(rec['gt_mask']).astype(np.uint8),(w,h),interpolation=cv2.INTER_NEAREST).astype(bool)
        r['flood_source']='gt_mask'
    else:
        wn=waternet_mask(pil)
        S['waternet_frac']=float(wn.mean())
        S['flood_teacher_iou']=float((wn&seg_water).sum()/max(1,(wn|seg_water).sum()))
        if rec['source']=='negative':
            # Curated no-flood image: the flood label is empty by source, NOT by deleting a teacher prediction.
            r['flood_source']='negative_source'
            if S['flood_teacher_iou']>=AGREEMENT_IOU and S['waternet_frac']>=NEG_REJECT_WATER_FRAC:
                r['reasons'].append('negative_source_but_teachers_agree_water')
            elif max(S['waternet_frac'],S['segformer_water_frac'])>NEG_WARN_WATER_FRAC:
                r['warnings'].append('teacher_water_on_negative')
        elif not wn.any() and not seg_water.any():
            if ALLOW_TEACHER_NEGATIVES_FROM_FLOOD_SOURCES: r['flood_source']='teachers_empty'
            else: r['reasons'].append('teachers_found_no_water_on_flood_source')
        elif S['flood_teacher_iou']<AGREEMENT_IOU:
            r['reasons'].append(f'water_teacher_iou_below_{AGREEMENT_IOU}')
        else:
            flood=wn; r['flood_source']='waternet_agreed'
    flood=fill_small_holes(flood,REGION_FILL_HOLE_FRAC*w*h)
    flood_polys=mask_to_polygons(flood,MIN_REGION_FRAC)
    if flood.any() and not flood_polys: r['reasons'].append('flood_regions_below_0.5_percent')
    flood_lbl=rasterize(flood_polys,h,w)
    S['flood_frac']=float(flood_lbl.mean())
    S['natural_water_in_flood_frac']=float((natural&flood).sum()/max(1,flood.sum())) if flood.any() else 0.0
    if rec['split']!='test' or STREET_CONTEXT_FILTER_TEST:
        if S['street_context_frac']<STREET_CONTEXT_MIN_FRAC: r['reasons'].append('no_street_context')
        elif flood.any() and S['natural_water_in_flood_frac']>=NATURAL_WATER_MAX_FRAC and S['street_context_frac']<STREET_CONTEXT_STRONG_FRAC:
            r['reasons'].append('natural_water_body')
    view={'rgb':np.asarray(pil),'flood_lbl':flood_lbl,'wn':wn,'seg_water':seg_water,'road_lbl':np.zeros((h,w),bool),
          'road_rejected':np.zeros((h,w),bool),'vehicle':np.zeros((h,w),bool),'car_polys':[]}
    r['view']=view
    if r['reasons']: r['status']='rejected'; return r
    car_rows,vehicle,dets=car_teacher(src,w,h); r['dets']=dets; view['vehicle']=vehicle
    view['car_polys']=[(int(x.split(' ',1)[0]),np.array(x.split()[1:],np.float32).reshape(-1,2)*[w,h]) for x in car_rows]
    road_polys,road_dbg,road_masks=analyze_road(road_argmax,road_prob,water_prob,flood,flood_lbl,vehicle)
    r['road']=road_dbg; road_dbg['action']='exported' if road_polys else 'none'
    view['road_rejected']=road_masks['rejected']|(road_masks['kept'] if road_dbg['status']=='unreliable' else False)
    if road_dbg['status']=='unreliable':
        if ROAD_UNRELIABLE_POLICY=='drop_image' and rec['split']!='test':
            road_dbg['action']='image_dropped'; r['reasons'].append('road_unreliable'); r['status']='rejected'; return r
        road_dbg['action']='kept_without_road'
    view['road_lbl']=rasterize(road_polys,h,w)
    rows=[f'0 {poly_to_row(p,w,h)}' for p in flood_polys]
    pseudo=[f'1 {poly_to_row(p,w,h)}' for p in road_polys]+car_rows
    if rec['split']=='test':
        # Test = WaterDataset flood ground truth only; road/car pseudo-labels are not ground truth.
        r['test_pseudo_not_exported']=collections.Counter(int(x.split(' ',1)[0]) for x in pseudo)
        if road_polys: road_dbg['action']='test_not_exported'
        view['road_lbl'][:]=False; view['car_polys']=[]
    else:
        rows+=pseudo
    r['rows']=rows
    return r

# ---------- Preview panel: original | label overlay / road mask | flood mask + text ----------
COLORS=[(0,0,255),(255,200,0),(0,220,0),(0,180,255),(255,0,180),(180,0,255),(255,255,0)]   # BGR per class
def _fit(img): return cv2.resize(img,(PANEL_W,max(1,round(img.shape[0]*PANEL_W/img.shape[1]))),interpolation=cv2.INTER_AREA)
def _caption(img,text):
    cv2.rectangle(img,(0,0),(img.shape[1],20),(0,0,0),-1); cv2.putText(img,text,(4,15),cv2.FONT_HERSHEY_SIMPLEX,.45,(255,255,255),1,cv2.LINE_AA); return img
def render_panel(view,lines,exported=True):
    base=cv2.cvtColor(view['rgb'],cv2.COLOR_RGB2BGR); h,w=base.shape[:2]
    ov=base.copy(); ov[view['flood_lbl']]=COLORS[0]; ov[view['road_lbl']]=COLORS[1]
    for cls,pts in view['car_polys']: cv2.fillPoly(ov,[np.round(pts).astype(np.int32).reshape(-1,1,2)],COLORS[cls])
    overlay=cv2.addWeighted(ov,0.45,base,0.55,0)
    for cls,pts in view['car_polys']:
        p=np.round(pts).astype(np.int32); cv2.polylines(overlay,[p.reshape(-1,1,2)],True,COLORS[cls],2)
        cv2.putText(overlay,CLASS_NAMES[cls],tuple(int(v) for v in p.min(0)),cv2.FONT_HERSHEY_SIMPLEX,max(.5,w/1400),COLORS[cls],2)
    rm=np.zeros_like(base); rm[view['road_rejected']]=(90,90,90); rm[view['vehicle']]=(140,70,0); rm[view['road_lbl']]=(255,255,255)
    fm=np.zeros_like(base); fl=view['flood_lbl']; sw=view['seg_water']
    fm[fl]=(255,255,255); fm[fl&~sw]=(150,150,255); fm[sw&~fl]=(255,255,0)
    if view['wn'] is not None: fm[view['wn']&~fl&~sw]=(0,165,255)
    tiles=[_caption(_fit(base),'original'),_caption(_fit(overlay),'exported labels: flood red, road cyan, car_Lk' if exported else 'candidate labels - NOT EXPORTED'),
           _caption(_fit(rm),'road label white | rejected road grey | vehicles blue'),
           _caption(_fit(fm),'flood label white | label&~SegF lightred | SegF only cyan | WaterNet only orange')]
    grid=np.vstack([np.hstack(tiles[:2]),np.hstack(tiles[2:])])
    lines=[part for t in lines for part in (textwrap.wrap(t,150) or [''])]
    lh=18; strip=np.zeros((lh*len(lines)+8,grid.shape[1],3),np.uint8)
    for i,t in enumerate(lines): cv2.putText(strip,t,(6,lh*(i+1)),cv2.FONT_HERSHEY_SIMPLEX,.45,(255,255,255),1,cv2.LINE_AA)
    return cv2.imencode('.jpg',np.vstack([grid,strip]),[cv2.IMWRITE_JPEG_QUALITY,85])[1].tobytes()

def describe(rec,r,uid):
    S=r['scores']; f=lambda k:f"{S[k]:.3f}" if k in S else '-'
    lines=[f"{uid} | {rec['source']} | group={rec['group'][:60]} | split={rec['split'] or 'train/val'} | {r['status'].upper()} {','.join(r['reasons'])}",
           f"flood: src={r.get('flood_source','-')} area={f('flood_frac')} teacherIoU={f('flood_teacher_iou')} WaterNet={f('waternet_frac')} SegFwater={f('segformer_water_frac')} natural_in_flood={f('natural_water_in_flood_frac')} street_ctx={f('street_context_frac')}"]
    d=r['road']
    if d: lines.append(f"road: {d['status']}/{d['action']} conf={d['kept_conf']:.2f} area={d['kept_frac']*100:.1f}% regions={d['n_kept']} (small {d['n_rejected_small']}, lowconf {d['n_rejected_low_conf']}) centroid_y={d['centroid_y']:.2f} top={d['top_band_share']:.2f} raw={d['raw_frac']*100:.1f}% P(road)lower={d['lower_road_prob']:.2f} uncertain={d['uncertain_frac']*100:.1f}% reasons={','.join(d['reasons']) or '-'}")
    else: lines.append('road: not evaluated (image rejected before road step)')
    if r['dets']: lines.append('cars: '+' '.join(f"L{x['level']}:{x['conf']:.2f}/iou{x['iou']:.2f}/{x['reason']}" for x in r['dets'][:8]))
    if r['warnings']: lines.append('warnings: '+','.join(r['warnings']))
    lines.append('NOTE: polygons are teacher pseudo-labels; a clean polygon is not evidence the label is correct.')
    return lines

PREVIEW_BANK=collections.defaultdict(list); bank_seen=collections.Counter(); bank_rng=random.Random(SEED)
def bank_offer(cat,uid,get_bytes):
    # Reservoir sampling: a uniform sample of PREVIEW_PER_CATEGORY images per category.
    bank_seen[cat]+=1
    if len(PREVIEW_BANK[cat])<PREVIEW_PER_CATEGORY: PREVIEW_BANK[cat].append((uid,get_bytes()))
    else:
        j=bank_rng.randrange(bank_seen[cat])
        if j<PREVIEW_PER_CATEGORY: PREVIEW_BANK[cat][j]=(uid,get_bytes())

results=[]; image_durations=[]; durations_by_source=collections.defaultdict(list)
for idx,rec in enumerate(records):
    t0=time.perf_counter(); src=rec['image']
    uid=hashlib.sha1((str(src)+str(rec['gt_mask'])).encode()).hexdigest()[:16]
    try:
        r=process_image(rec)
    except Exception as e:
        print(f'[DROP] {src}: {type(e).__name__}: {e}')
        r={'status':'rejected','reasons':['teacher_or_decode_error'],'warnings':[],'rows':[],'road':None,'dets':[],'scores':{},'error':f'{type(e).__name__}: {e}'}
    view=r.pop('view',None); r.update(record=rec,uid=uid,split=rec['split'],group=rec['group'])
    counts=collections.Counter(int(x.split(' ',1)[0]) for x in r['rows']); r['counts']=counts
    r['flood_positive']=counts[0]>0
    if view is not None:
        cache={}
        def get_bytes():
            if 'b' not in cache: cache['b']=render_panel(view,describe(rec,r,uid),exported=r['status']=='kept')
            return cache['b']
        d=r['road']
        if r['status']=='kept' and counts[1]>0: (PREVIEW_ROAD/f'{uid}.jpg').write_bytes(get_bytes())
        if d and d['status']=='unreliable': (PREVIEW_ROAD_REJECTED/f'{uid}.jpg').write_bytes(get_bytes())
        if r['status']=='rejected': bank_offer('rejected_'+r['reasons'][0].split('_below_')[0],uid,get_bytes)
        elif rec['split']=='test': bank_offer('test_flood_gt',uid,get_bytes)
        else:
            if counts[0] and not counts[1] and not any(counts[c] for c in range(2,7)): bank_offer('flood_only',uid,get_bytes)
            if counts[1]: bank_offer('road',uid,get_bytes)
            for c in range(2,7):
                if counts[c]: bank_offer(CLASS_NAMES[c],uid,get_bytes)
            if not counts[0]: bank_offer('negative_empty_label' if not r['rows'] else 'nonflood_with_road_or_car',uid,get_bytes)
    results.append(r)
    elapsed=time.perf_counter()-t0; image_durations.append(elapsed); durations_by_source[rec['source']].append(elapsed)
    if (idx+1)%10==0:
        st=collections.Counter(x['status'] for x in results)
        print(f'[PROGRESS] {idx+1}/{len(records)} images; kept={st["kept"]}; rejected={st["rejected"]}; avg={np.mean(image_durations):.2f}s/image')
kept_results=[r for r in results if r['status']=='kept']
print('Kept:',len(kept_results),'| rejected by first reason:',dict(collections.Counter(r['reasons'][0] for r in results if r['status']=='rejected')))
print('Road status/action:',dict(collections.Counter((r['road']['status'],r['road']['action']) for r in results if r['road'])))
for source in sorted(full_source_counts):
    vals=durations_by_source.get(source,[])
    avg=float(np.mean(vals)) if vals else 0.0
    print(f'[TIME] {source}: {avg:.2f}s/image average; estimated full source {full_source_counts[source]*avg/60:.1f} min for {full_source_counts[source]} images')
print(f'[TIME] Overall average {np.mean(image_durations):.2f}s/image across {len(image_durations)} sampled images' if image_durations else '[TIME] No images processed')


## 7. Group split train/val có ràng buộc class, cân bằng ảnh không flood, ghi YOLO dataset

Tìm ngẫu nhiên `SPLIT_SEARCH_TRIALS` cách chia group; chọn cách mà mọi class có ở ≥ 2 group đều có trong train **và** val, class chỉ ở 1 group nằm trong train, có ảnh không flood ở cả hai (và cân tỉ lệ), gần `VAL_FRACTION` nhất. Vi phạm còn lại được in ra và làm dataset "chưa sẵn sàng". Ảnh không flood mỗi split được cắt về `NONFLOOD_TARGET_FRAC` (≤ 20%); thiếu so với `NONFLOOD_MIN_FRAC` thì báo số ảnh thiếu.


In [ ]:
# ---------- Group split train/val with class-presence constraints ----------
# Random search over group orders: val = first groups until VAL_FRACTION of images. A candidate is penalised when
# a required class (present in >= SPLIT_MIN_GROUPS_FOR_CLASS groups) is missing from train or val, when a class
# that lives in a single group ends up in val (it must be in train), when non-flood images are missing, and by the
# distance to VAL_FRACTION / per-class balance. The best candidate is kept and its constraint status reported.
trainval=[r for r in kept_results if r['split']!='test']
tests=[r for r in kept_results if r['split']=='test']
NC=len(CLASS_NAMES)
gnames=sorted({r['record']['group'] for r in trainval}); gi={g:i for i,g in enumerate(gnames)}
G=len(gnames)
g_img=np.zeros(G); g_nonflood=np.zeros(G); g_cls_img=np.zeros((G,NC)); g_cls_inst=np.zeros((G,NC))
for r in trainval:
    i=gi[r['record']['group']]; g_img[i]+=1; g_nonflood[i]+=not r['flood_positive']
    for c,n in r['counts'].items(): g_cls_img[i,c]+=1; g_cls_inst[i,c]+=n
cls_groups=(g_cls_img>0).sum(0)
required=[c for c in range(NC) if cls_groups[c]>=SPLIT_MIN_GROUPS_FOR_CLASS]
single_group=[c for c in range(NC) if cls_groups[c]==1]
split_constraints={'required_in_train_and_val':[CLASS_NAMES[c] for c in required],
                   'single_group_forced_train':[CLASS_NAMES[c] for c in single_group],
                   'absent':[CLASS_NAMES[c] for c in range(NC) if cls_groups[c]==0]}
print('Split constraints:',split_constraints)

def evaluate(val_mask):
    tr=~val_mask; pen=0.0; issues=[]
    tr_img=g_cls_img[tr].sum(0); va_img=g_cls_img[val_mask].sum(0)
    for c in required:
        if tr_img[c]==0: pen+=1000; issues.append(f'{CLASS_NAMES[c]} missing in train')
        if va_img[c]==0: pen+=100; issues.append(f'{CLASS_NAMES[c]} missing in val')
    for c in single_group:
        if va_img[c]>0: pen+=1000; issues.append(f'{CLASS_NAMES[c]} (1 group) in val instead of train')
    if (g_nonflood>0).sum()>=2:
        if g_nonflood[tr].sum()==0: pen+=50; issues.append('no non-flood image in train')
        if g_nonflood[val_mask].sum()==0: pen+=10; issues.append('no non-flood image in val')
    frac=g_img[val_mask].sum()/max(1,g_img.sum()); pen+=100*abs(frac-VAL_FRACTION)
    inst=g_cls_inst.sum(0)
    for c in required: pen+=5*abs(g_cls_inst[val_mask,c].sum()/max(1,inst[c])-VAL_FRACTION)
    if g_nonflood.sum(): pen+=5*abs(g_nonflood[val_mask].sum()/g_nonflood.sum()-VAL_FRACTION)
    return pen,issues,frac

if G<2: raise ValueError(f'Chỉ có {G} group cho train/val; cần ít nhất 2.')
split_rng=np.random.default_rng(SEED); target=VAL_FRACTION*g_img.sum(); best=None
for t in range(SPLIT_SEARCH_TRIALS):
    order=split_rng.permutation(G)
    k=int(np.searchsorted(np.cumsum(g_img[order]),target))+1
    k=min(max(k,1),G-1)
    val_mask=np.zeros(G,bool); val_mask[order[:k]]=True
    pen,issues,frac=evaluate(val_mask)
    if best is None or pen<best[0]: best=(pen,issues,frac,val_mask)
pen,split_issues,val_frac,val_mask=best
val_groups={gnames[i] for i in np.flatnonzero(val_mask)}
for r in trainval: r['split']='val' if r['record']['group'] in val_groups else 'train'
print(f'Split: val={val_frac:.1%} ảnh ({len(val_groups)}/{G} group); penalty={pen:.1f}; vi phạm: {split_issues or "không"}')

# ---------- Non-flood share per split: cap at NONFLOOD_TARGET_FRAC, report deficit below NONFLOOD_MIN_FRAC ----------
nonflood_report={}
for split in ('train','val'):
    items=[r for r in trainval if r['split']==split]
    pos=[r for r in items if r['flood_positive']]; non=[r for r in items if not r['flood_positive']]
    max_non=math.floor(NONFLOOD_TARGET_FRAC*len(pos)/(1-NONFLOOD_TARGET_FRAC))
    keep=choose_group_balanced(non,max_non,random.Random(SEED)) if len(non)>max_non else non
    keep_ids={r['uid'] for r in keep}
    for r in non:
        if r['uid'] not in keep_ids:
            r['status']='dropped_nonflood_cap'; r['reasons'].append(f'nonflood_cap_{NONFLOOD_TARGET_FRAC:.0%}')
    min_non=math.ceil(NONFLOOD_MIN_FRAC*len(pos)/(1-NONFLOOD_MIN_FRAC))
    nonflood_report[split]={'flood_positive':len(pos),'nonflood_available':len(non),'nonflood_kept':len(keep),
                            'nonflood_min_needed':min_non,'deficit':max(0,min_non-len(keep)),
                            'nonflood_frac':len(keep)/max(1,len(pos)+len(keep)),
                            'empty_label':sum(1 for r in keep if not r['rows'])}
print('Non-flood per split:',nonflood_report)

final=[r for r in kept_results if r['status']=='kept']
assert not ({r['record']['group'] for r in final if r['split']=='train'}&{r['record']['group'] for r in final if r['split']=='val'}), 'group leakage train/val'
assert not ({r['record']['group'] for r in final if r['split']=='test'}&{r['record']['group'] for r in final if r['split']!='test'}), 'group leakage test'
for r in final:
    split=r['split']
    Image.open(r['record']['image']).convert('RGB').save(OUT/'images'/split/(r['uid']+'.jpg'),quality=95)
    (OUT/'labels'/split/(r['uid']+'.txt')).write_text(('\n'.join(r['rows'])+'\n') if r['rows'] else '')

import yaml
with (OUT/'data.yaml').open('w') as f:
    yaml.safe_dump({'path':str(OUT.resolve()),'train':'images/train','val':'images/val','test':'images/test','names':{i:n for i,n in enumerate(CLASS_NAMES)}},f,sort_keys=False,allow_unicode=True)
print('Written:',dict(collections.Counter(r['split'] for r in final)))


## 8. Thống kê theo split, preview cân bằng, cổng chất lượng và manifest

- Bảng số ảnh (flood-positive / không flood có road-car / nhãn rỗng) và số instance từng class theo split; thống kê xe từng level và road.
- `/kaggle/working/preview/<category>/`: tối đa `PREVIEW_PER_CATEGORY` ảnh cho mỗi nhóm `flood_only`, `road`, `car_L0…car_L4`, `negative_empty_label`, `nonflood_with_road_or_car`, `test_flood_gt`, `rejected_<lý do>`; `preview/README.txt` ghi rõ nhóm không có mẫu.
- `/kaggle/working/auto_label_manifest.csv`: mọi ảnh ứng viên với source, group, split, giữ/bỏ + lý do, số instance, điểm teacher, confidence road, detection xe.
- `dataset/QUALITY_REPORT.json` và `READY_FOR_TRAINING.txt` / `NOT_READY_FOR_TRAINING.txt`.
- Cell dừng với lỗi nếu nhãn road đã ghi chồng flood hoặc test có nhãn khác flood.


In [ ]:
SPLITS=('train','val','test')
final=[r for r in results if r['status']=='kept']
by_uid={r['uid']:r for r in results}

# ---------- Safety net: re-read written labels; road must not overlap flood; test must be flood-only ----------
overlap_bad=[]; test_bad=[]
for r in final:
    w,h=r['size']; polys=collections.defaultdict(list)
    for line in (OUT/'labels'/r['split']/(r['uid']+'.txt')).read_text().splitlines():
        vals=line.split(); polys[int(vals[0])].append(np.round(np.array(vals[1:],np.float32).reshape(-1,2)*[w,h]))
    if r['split']=='test' and any(c!=0 for c in polys): test_bad.append(r['uid'])
    if polys[0] and polys[1]:
        ov=int((rasterize(polys[0],h,w)&rasterize(polys[1],h,w)).sum())
        if ov: overlap_bad.append((r['uid'],ov))
print(f'[CHECK] road/flood overlap in written labels: {len(overlap_bad)} ảnh {overlap_bad[:10]} | test có nhãn khác flood: {len(test_bad)}')

# ---------- Tables per split ----------
def split_table():
    rows=[]
    for s in SPLITS:
        items=[r for r in final if r['split']==s]
        inst=collections.Counter(); imgs=collections.Counter()
        for r in items:
            for c,n in r['counts'].items(): inst[c]+=n; imgs[c]+=1
        rows.append({'split':s,'images':len(items),'flood_positive':sum(r['flood_positive'] for r in items),
                     'nonflood_road_or_car':sum((not r['flood_positive']) and bool(r['rows']) for r in items),
                     'empty_label':sum(not r['rows'] for r in items),
                     **{f'inst_{n}':inst[c] for c,n in enumerate(CLASS_NAMES)},**{f'img_{n}':imgs[c] for c,n in enumerate(CLASS_NAMES)}})
    return rows
table=split_table()
hdr=['split','images','flood_positive','nonflood_road_or_car','empty_label']+[f'inst_{n}' for n in CLASS_NAMES]
print('\n'+' | '.join(f'{x:>9}' if i else f'{x:<6}' for i,x in enumerate(['split','images','flood+','nonflood','empty']+CLASS_NAMES)))
for row in table: print(' | '.join(f'{row[k]:>9}' if i else f'{row[k]:<6}' for i,k in enumerate(hdr)))
print('(số instance; ảnh test chỉ có flood GT - road/car pseudo không được ghi vào test)')
print('Ảnh chứa class theo split:',{row['split']:{n:row[f'img_{n}'] for n in CLASS_NAMES} for row in table})
T={row['split']:row for row in table}

# ---------- Car statistics per level ----------
car_stats={k:collections.Counter() for k in range(5)}; car_other=collections.Counter()
for r in results:
    for d in r['dets']:
        if d['level'] is None: car_other[d['reason']]+=1; continue
        st=car_stats[d['level']]; st['teacher_detections_conf>=0.5']+=1
        if r['status']!='kept': st['image_not_kept']+=1
        elif d['matched'] and r['split']=='test': st['test_pseudo_not_exported']+=1
        elif d['matched']: st['exported']+=1
        else: st['dropped_'+d['reason']]+=1
print('\nCar theo level (detection của teacher, không phải instance polygon):')
for k in range(5): print(f'  car_L{k}:',dict(car_stats[k]) or 'teacher KHÔNG phát hiện mẫu nào')
if car_other: print('  detection không đọc được level:',dict(car_other))

# ---------- Road statistics ----------
road_rows=[r for r in results if r['road']]
print('\nRoad status/action:',dict(collections.Counter((r['road']['status'],r['road']['action']) for r in road_rows)))
print('Lý do road unreliable:',dict(collections.Counter(x for r in road_rows for x in r['road']['reasons'])))
unreliable_rate=sum(r['road']['status']=='unreliable' for r in road_rows)/max(1,len(road_rows))
ok=[r['road'] for r in road_rows if r['road']['status']=='ok']
for key in ('kept_conf','kept_frac','n_kept','centroid_y','uncertain_frac'):
    if ok:
        v=np.array([d[key] for d in ok],float); print(f'  road ok {key}: p10={np.percentile(v,10):.3f} median={np.median(v):.3f} p90={np.percentile(v,90):.3f}')
n_road_preview=len(list(PREVIEW_ROAD.glob('*.jpg')))
n_road_images=sum(r['counts'][1]>0 for r in final)
n_road_candidates=sum(r['counts'][1]>0 for r in results if r['status'] in ('kept','dropped_nonflood_cap'))   # panels are written before the non-flood cap
print(f'Preview road: {n_road_preview} panel cho ảnh có road (ảnh có road cuối cùng: {n_road_images}) tại {PREVIEW_ROAD}; road unreliable: {len(list(PREVIEW_ROAD_REJECTED.glob("*.jpg")))} tại {PREVIEW_ROAD_REJECTED}')

# ---------- Balanced previews ----------
EXPECTED_CATS=['flood_only','road']+CLASS_NAMES[2:]+['negative_empty_label','nonflood_with_road_or_car','test_flood_gt']
cats=EXPECTED_CATS+sorted(c for c in PREVIEW_BANK if c not in EXPECTED_CATS)
readme=[]
for cat in cats:
    items=PREVIEW_BANK.get(cat,[])
    if not items: readme.append(f'{cat}: KHÔNG CÓ MẪU'); continue
    d=PREVIEW/cat; d.mkdir(parents=True,exist_ok=True)
    for uid,b in items:
        img=cv2.imdecode(np.frombuffer(b,np.uint8),cv2.IMREAD_COLOR); r=by_uid[uid]
        bar=np.zeros((24,img.shape[1],3),np.uint8); bar[:]=(0,0,160) if r['status']!='kept' else (0,110,0)
        cv2.putText(bar,f"FINAL: {r['status']} | split={r['split'] or '-'} | {';'.join(r['reasons']) or 'kept'}",(6,17),cv2.FONT_HERSHEY_SIMPLEX,.5,(255,255,255),1,cv2.LINE_AA)
        img=np.vstack([img,bar])
        cv2.imwrite(str(d/f'{uid}.jpg'),img)
    readme.append(f'{cat}: {len(items)} preview / {bank_seen[cat]} ảnh')
(PREVIEW/'README.txt').write_text('\n'.join(readme)+'\n')
print('\nPreview cân bằng:',*readme,sep='\n  ')

# ---------- Quality gate ----------
problems=[]; warnings_=[]
if overlap_bad: problems.append(f'nhãn road chồng flood ở {len(overlap_bad)} ảnh')
if test_bad: problems.append(f'{len(test_bad)} ảnh test có nhãn khác flood')
for s,mins in QUALITY_MIN_IMAGES.items():
    for cls,m in mins.items():
        if T[s][f'img_{cls}']<m: problems.append(f'{s}: chỉ {T[s][f"img_{cls}"]} ảnh có {cls} (< {m})')
for c in CLASS_NAMES[2:]:
    total=sum(T[s][f'inst_{c}'] for s in ('train','val'))
    if total==0: problems.append(f'{c}: không có instance nào (teacher không phát hiện/ghép được) - THIẾU DỮ LIỆU, không tự tạo')
    elif T['train'][f'inst_{c}']==0: problems.append(f'{c}: có {total} instance nhưng KHÔNG có trong train')
    elif T['train'][f'inst_{c}']<QUALITY_MIN_TRAIN_CAR_INSTANCES: warnings_.append(f'{c}: chỉ {T["train"][f"inst_{c}"]} instance trong train')
for cls in ('flood','road'):
    if T['train'][f'inst_{cls}']==0: problems.append(f'{cls} KHÔNG có trong train')
for msg in split_issues: problems.append('split: '+msg)
for s,rep in nonflood_report.items():
    if rep['deficit']>0 or rep['nonflood_frac']<NONFLOOD_MIN_FRAC:
        problems.append(f"{s}: ảnh không flood {rep['nonflood_kept']}/{rep['flood_positive']+rep['nonflood_kept']} = {rep['nonflood_frac']:.1%} < {NONFLOOD_MIN_FRAC:.0%}; THIẾU {rep['deficit']} ảnh")
    assert rep['nonflood_frac']<=NONFLOOD_MAX_FRAC+1e-9
if T['train']['empty_label']==0: warnings_.append('train không có ảnh nhãn rỗng (ảnh âm nào cũng có road/xe): model chưa thấy khung hình không có class nào')
neg_kept=collections.Counter(r['record']['neg_category'] for r in final if r['record']['source']=='negative')
missing_cats=[c for c in NEGATIVE_CATEGORIES if neg_kept.get(c,0)==0]
if missing_cats: warnings_.append(f'thiếu ảnh âm loại {missing_cats} (theo tên thư mục)')
if T['test']['images']<QUALITY_MIN_TEST_IMAGES: warnings_.append(f"test chỉ có {T['test']['images']} ảnh (< {QUALITY_MIN_TEST_IMAGES})")
if unreliable_rate>ROAD_MAX_UNRELIABLE_RATE: warnings_.append(f'{unreliable_rate:.0%} ảnh có road unreliable: ngưỡng hoặc SEGFORMER_MODEL không hợp dữ liệu, xem preview_road_rejected/')
if n_road_preview!=n_road_candidates: problems.append(f'preview road ({n_road_preview}) không khớp số ảnh có road ({n_road_candidates})')
if LIMIT is not None: warnings_.append(f'LIMIT={LIMIT}: chạy thử, không phải dataset đầy đủ')
need_neg=sum(rep['deficit'] for rep in nonflood_report.values())
if need_neg: print(f'\n[THIẾU ẢNH ÂM] Cần thêm ít nhất {need_neg} ảnh không ngập (train+val, sau lọc bối cảnh). Nguồn cần bổ sung: '
                   'đường khô, đường ướt sau mưa không ngập, phản chiếu trên kính/gương/mặt đường, vũng nước nhỏ - ảnh đường phố '
                   'cùng góc nhìn với dữ liệu ngập (dashcam/CCTV/điện thoại). Đặt trong thư mục non_flooded/, dry_road/, wet_road/, '
                   'reflection/, puddle/ hoặc khai báo NEGATIVE_INPUT_DIRS. Notebook KHÔNG tạo ảnh âm bằng cách xoá nhãn flood.')
ready=not problems
report={'ready_for_training':ready,'problems':problems,'warnings':warnings_,'limit':LIMIT,'split_table':table,
        'nonflood':nonflood_report,'split_constraints':split_constraints,'car_levels':{f'car_L{k}':dict(v) for k,v in car_stats.items()},
        'road_status':{f'{a}/{b}':n for (a,b),n in collections.Counter((r['road']['status'],r['road']['action']) for r in road_rows).items()},
        'rejected':dict(collections.Counter(r['reasons'][0] for r in results if r['status']!='kept')),
        'negative_categories_kept':dict(neg_kept),
        'notes':['test = WaterDataset flood ground truth only; road/car in test are not labelled (pseudo-labels are not GT)',
                 'flood (non-GT), road and car are teacher pseudo-labels; review previews before training']}
(OUT/'QUALITY_REPORT.json').write_text(json.dumps(report,ensure_ascii=False,indent=1,default=str))
for name in ('READY_FOR_TRAINING.txt','NOT_READY_FOR_TRAINING.txt'):
    if (OUT/name).exists(): (OUT/name).unlink()
if ready: (OUT/'READY_FOR_TRAINING.txt').write_text('Tất cả kiểm tra tự động đạt. Vẫn cần review preview.\n'+'\n'.join('WARNING: '+x for x in warnings_)+'\n')
else: (OUT/'NOT_READY_FOR_TRAINING.txt').write_text('Dataset CHƯA sẵn sàng để train:\n'+'\n'.join('- '+x for x in problems)+'\n\nCảnh báo:\n'+'\n'.join('- '+x for x in warnings_)+'\n')

# ---------- Manifest: every candidate image with source/group/split/decision/scores ----------
with (WORK/'auto_label_manifest.csv').open('w',newline='') as f:
    wr=csv.writer(f)
    wr.writerow(['uid','image','source','neg_category','group','split','status','reasons','warnings','flood_source','flood_positive','label_empty']+
                [f'n_{n}' for n in CLASS_NAMES]+['flood_frac','flood_teacher_iou','waternet_frac','segformer_water_frac','street_context_frac',
                'natural_water_in_flood_frac','road_status','road_action','road_reasons','road_conf','road_frac','road_regions','road_centroid_y',
                'road_top_band_share','road_uncertain_frac','road_raw_frac','road_lower_prob','car_detections','test_pseudo_not_exported','gt_mask','error'])
    for r in results:
        rec=r['record']; S=r['scores']; d=r['road'] or {}; sc=lambda k:round(S[k],4) if k in S else ''
        rd=lambda k:round(d[k],4) if k in d else ''
        wr.writerow([r['uid'],rec['image'],rec['source'],rec['neg_category'] or '',rec['group'],r['split'] or '',r['status'],';'.join(r['reasons']),
                     ';'.join(r['warnings']),r.get('flood_source',''),r['flood_positive'],r['status']=='kept' and not r['rows']]+
                    [r['counts'][c] for c in range(NC)]+[sc('flood_frac'),sc('flood_teacher_iou'),sc('waternet_frac'),sc('segformer_water_frac'),
                     sc('street_context_frac'),sc('natural_water_in_flood_frac'),d.get('status',''),d.get('action',''),';'.join(d.get('reasons',[])),
                     rd('kept_conf'),rd('kept_frac'),d.get('n_kept',''),rd('centroid_y'),rd('top_band_share'),rd('uncertain_frac'),rd('raw_frac'),rd('lower_road_prob'),
                     json.dumps(r['dets']),json.dumps({CLASS_NAMES[c]:n for c,n in r.get('test_pseudo_not_exported',{}).items()}),rec['gt_mask'] or '',r.get('error','')])
print('Manifest:',WORK/'auto_label_manifest.csv','| report:',OUT/'QUALITY_REPORT.json')

print('\n'+'='*80)
print('DATASET READY FOR TRAINING' if ready else 'DATASET CHƯA SẴN SÀNG ĐỂ TRAIN (xem NOT_READY_FOR_TRAINING.txt)')
for x in problems: print('  [PROBLEM]',x)
for x in warnings_: print('  [WARNING]',x)
print('='*80)
if overlap_bad or test_bad: raise RuntimeError('Nhãn đã ghi vi phạm road/flood hoặc test flood-only; xem [CHECK] ở trên.')


## 9. Phân tích manifest (ảnh bị loại ở đâu)

Crosstab nguồn × trạng thái, nguồn × lý do loại, lý do road unreliable, và median điểm theo lý do — dùng để chọn ngưỡng cần nới hoặc nguồn cần bổ sung.

In [ ]:
# Phân tích manifest: ảnh bị loại ở đâu và vì sao (chạy sau cell 8, hoặc riêng khi đã có auto_label_manifest.csv).
import pandas as pd
pd.set_option('display.width',200); pd.set_option('display.max_columns',50); pd.set_option('display.max_rows',200)
# Session edit mới có /kaggle/working trống: add Output của version đã chạy full (Add Input → Your Work → Notebooks)
# thì manifest nằm dưới /kaggle/input/<tên-notebook>/. Ưu tiên file ở working (vừa chạy xong), sau đó file mới nhất ở input.
from pathlib import Path
cands=[p for p in [Path('/kaggle/working/auto_label_manifest.csv')] if p.exists()] or \
      sorted(Path('/kaggle/input').rglob('auto_label_manifest.csv'),key=lambda p:p.stat().st_mtime,reverse=True)
if not cands: raise FileNotFoundError('Không thấy auto_label_manifest.csv trong /kaggle/working hay /kaggle/input. '
                                      'Chạy hết notebook, hoặc add Output của version đã chạy full làm Input.')
print('Manifest:',cands[0])
m=pd.read_csv(cands[0],keep_default_na=False)
m['first_reason']=m['reasons'].str.split(';').str[0].replace('','-')
print(f'Tổng ứng viên: {len(m)} | kept: {(m.status=="kept").sum()} | rejected: {(m.status!="kept").sum()}')

print('\n=== Nguồn × trạng thái ===')
print(pd.crosstab(m['source'],m['status'],margins=True))
print('\n=== Nguồn × lý do loại đầu tiên (chỉ ảnh bị loại) ===')
rej=m[m.status!='kept']
print(pd.crosstab(rej['source'],rej['first_reason'],margins=True).T)
print('\n=== Tất cả lý do (một ảnh có thể nhiều lý do) ===')
print(rej['reasons'].str.split(';').explode().value_counts())

print('\n=== Lý do road unreliable (ảnh bị loại vì road_unreliable) ===')
rr=m[m['reasons'].str.contains('road_unreliable')]
if len(rr):
    print(pd.crosstab(rr['road_reasons'].str.split(';').explode(),rr.loc[rr.index.repeat(rr['road_reasons'].str.split(';').str.len()),'source'].values))
print('\n=== Kept theo nguồn × split ===')
k=m[m.status=='kept']
print(pd.crosstab(k['source'],k['split'].replace('','-'),margins=True))
if (m['neg_category']!='').any():
    print('\n=== Ảnh âm theo loại × trạng thái ===')
    n=m[m['neg_category']!='']; print(pd.crosstab(n['neg_category'],n['status'],margins=True))

# Phân phối điểm của ảnh bị loại theo lý do chính, để biết nới ngưỡng nào có lợi nhất.
num=['flood_teacher_iou','street_context_frac','natural_water_in_flood_frac','road_conf','road_frac','road_raw_frac','road_lower_prob','road_uncertain_frac']
for c in num: m[c]=pd.to_numeric(m[c],errors='coerce')
print('\n=== Median điểm theo lý do loại đầu tiên ===')
print(m.groupby('first_reason')[num].median().round(3))


## Output

- `/kaggle/working/dataset`: YOLO-seg (`images/`, `labels/`, `data.yaml`), `QUALITY_REPORT.json`, và `READY_FOR_TRAINING.txt` hoặc `NOT_READY_FOR_TRAINING.txt`. **Chỉ train khi có `READY_FOR_TRAINING.txt`** và đã xem preview.
- `/kaggle/working/preview/`, `preview_road/`, `preview_road_rejected/`, `auto_label_manifest.csv`.

Test chỉ chứa group WaterDataset giữ riêng, chỉ có nhãn flood thật; dùng để đánh giá flood, không đánh giá road/car.
